# M28 · Matrice de Gram, montée sur l'entrée et lois tronquées — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m28_gram_echantillonnage/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m28_gram_echantillonnage/03_notebook.ipynb`, créée par `python tools/start_chapter.py M28`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M28.1 à M28.7, M28.11 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M28.13 | Une configuration de points de matrice de Gram donnée, par le théorème spectral · `points_from_gram` | 🔨 | ★★ | 25 |
|  | M28.14 | Tempérer une loi donnée par ses probabilités, et le plus petit ensemble de masse donnée · `temper`, `smallest_mass_set` | 🔨 | ★★ | 30 |
| B | M28.15 | Prédire ce que deviennent un pas η∇f et un pas normalisé quand la fonction est multipliée par 1 000, puis vérifier | 🔮 | ★ | 15 |
|  | M28.16 | Des motifs rayés de fréquence croissante jusqu'au damier : leur variation totale, et ce qu'en laissent un flou 3 × 3 et dix pas de descente sur la variation totale | 🎨 | ★ | 20 |
| C | M28.17 | Chasse à l'erreur : un double centrage fait sur les distances au lieu de leurs carrés, une montée qui coupe le gradient au lieu du point, une prépériode comptée dans la période | 🐛 | ★ | 20 |
|  | M28.18 | Un test de propriété : la configuration rendue redonne la matrice de Gram et les distances, pour des points tirés au hasard | 🛠️ | ★ | 15 |
| D | M28.19 | Défi : redessiner la carte de 30 villes quand un quart des distances manquent, à moins de 1 km en moyenne après alignement | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M28")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : une matrice de Gram et une matrice des moments, un double centrage et une carte, une montée projetée, la variation totale d'une barre, des étoffes rayées, une prévision refroidie et tronquée, des suites qui bouclent, et un pont vers la température sur des scores.
- Écrire les trois fonctions de `mymath/sampling.py` (`points_from_gram`, `temper`, `smallest_mass_set`) et les tester.
- Prédire ce qu'un changement d'unité fait à un pas de montée, dessiner des motifs rayés et leur lissage, corriger trois erreurs classiques, écrire un test de propriété, et relever un défi.

**Rappel express.** $\mathbf{G} = \mathbf{X}\mathbf{X}^\top$, $\lVert\mathbf{x}_i - \mathbf{x}_j\rVert^2 = G_{ii} + G_{jj} - 2G_{ij}$ ; pour des points centrés, $\mathbf{G} = -\frac12\mathbf{J}\mathbf{D}^{(2)}\mathbf{J}$ (les **carrés** des distances), et les colonnes $\sqrt{\lambda_k}\,\mathbf{q}_k$ donnent une carte à une isométrie près. Montée projetée : $\mathbf{x} \leftarrow \operatorname{clip}(\mathbf{x} + \eta\,\mathbf{s})$, on coupe le **point** ; le pas normalisé ne voit pas l'unité de $f$. $\mathrm{VT}(\mathbf{x}) = \mathbf{x}^\top\mathbf{L}\mathbf{x}$, gradient $2\mathbf{L}\mathbf{x}$ ; un motif de fréquence $f$ : rapport $4\sin^2(\pi f)$, gain $\frac{1 + 2\cos 2\pi f}{3}$. Température : $p_i^{1/T}$ renormalisé ; top-p : le plus petit ensemble de masse $\ge m$ ; $\mathrm{KL}(\mathbf{p}_S \,\|\, \mathbf{p}) = -\ln P(S)$. Période : la longueur du cycle, sans la queue.

## Partie 0 · Vérifier tes exercices papier (✏️ M28.1 à M28.7 ; 🔗 M28.11)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question (« a number » sans décimales : la valeur exacte, ces calculs tombent juste) ;
- une liste `[0.2, -1]`, dans l'ordre que dit la question ; un ensemble d'issues comme la liste de leurs numéros, `[4, 0, 2]` ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]` ;
- un intervalle entre guillemets, en notation française, avec des points décimaux, `"]2.5 ; 4]"` ;
- `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

**Ex M28.1 — Quatre points du plan : leur matrice de Gram 4 × 4, leur matrice des moments d'ordre 2, et les distances lues dans la Gram**

In [ ]:
# Ex M28.1: the values you found on paper
answer_M28_1a = ...  # a) a 4 × 4 matrix, as the list of its rows (m²)
answer_M28_1b = ...  # b) a 2 × 2 matrix, as the list of its rows (m²)
answer_M28_1c = ...  # c) a list of 2 numbers, in decreasing order
answer_M28_1d = ...  # d) a list of 3 numbers (m²)
answer_M28_1e = ...  # e) a number (1 decimal), in degrees

for letter, answer in zip("abcde", [answer_M28_1a, answer_M28_1b, answer_M28_1c, answer_M28_1d, answer_M28_1e]):
    wbm.check(f"M28.1{letter}", answer)

**Ex M28.2 — Quatre villes aux sommets d'un losange : le double centrage des carrés des distances, puis une carte à une isométrie près**

In [ ]:
# Ex M28.2: the values you found on paper
answer_M28_2a = ...  # a) a number (km²)
answer_M28_2b = ...  # b) a 4 × 4 matrix, as the list of its rows (km²)
answer_M28_2c = ...  # c) a list of 4 numbers, in decreasing order
answer_M28_2d = ...  # d) a 4 × 2 matrix, as the list of its rows (km)
answer_M28_2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_M28_2a, answer_M28_2b, answer_M28_2c, answer_M28_2d, answer_M28_2e]):
    wbm.check(f"M28.2{letter}", answer)

**Ex M28.3 — Régler quatre émetteurs entre 0 et 1 : le maximum d'une réponse linéaire sur la boule et sur le pavé, d'une énergie quadratique sur la sphère, et deux pas de montée projetée**

In [ ]:
# Ex M28.3: the values you found on paper
answer_M28_3a = ...  # a) a number
answer_M28_3b = ...  # b) a list of 4 numbers
answer_M28_3c = ...  # c) an integer
answer_M28_3d = ...  # d) a number
answer_M28_3e = ...  # e) a number
answer_M28_3f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M28_3a, answer_M28_3b, answer_M28_3c, answer_M28_3d, answer_M28_3e, answer_M28_3f]):
    wbm.check(f"M28.3{letter}", answer)

**Ex M28.4 — Les températures le long d'une barre : la variation totale xᵀLx, son gradient 2Lx, un pas de lissage, et une marche étalée sur n pas, au carré ou en valeur absolue**

In [ ]:
# Ex M28.4: the values you found on paper
answer_M28_4a = ...  # a) a number (°C²)
answer_M28_4b = ...  # b) a list of 5 numbers
answer_M28_4c = ...  # c) a list of 5 numbers (°C)
answer_M28_4d = ...  # d) a number (2 decimals) (°C²)
answer_M28_4e = ...  # e) a number (4 decimals)
answer_M28_4f = ...  # f) a list of 2 numbers: squared, then absolute value
answer_M28_4g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M28_4a, answer_M28_4b, answer_M28_4c, answer_M28_4d, answer_M28_4e, answer_M28_4f, answer_M28_4g]):
    wbm.check(f"M28.4{letter}", answer)

**Ex M28.5 — Des étoffes rayées : période, fréquence, fréquence de Nyquist, repliement, variation totale rapportée à l'énergie 4 sin²(ω/2), et gain d'une moyenne mobile**

In [ ]:
# Ex M28.5: the values you found on paper
answer_M28_5a = ...  # a) a list of 2 numbers (3 decimals)
answer_M28_5b = ...  # b) a list of 2 numbers (4 decimals)
answer_M28_5c = ...  # c) a number (2 decimals), in cells
answer_M28_5d = ...  # d) a number (3 decimals)
answer_M28_5e = ...  # e) a number (3 decimals)
answer_M28_5f = ...  # f) a number, in cells

for letter, answer in zip("abcdef", [answer_M28_5a, answer_M28_5b, answer_M28_5c, answer_M28_5d, answer_M28_5e, answer_M28_5f]):
    wbm.check(f"M28.5{letter}", answer)

**Ex M28.6 — La prévision d'un météorologue en six catégories : la refroidir, garder les trois plus probables, garder le plus petit ensemble de masse donnée, et le coût de chaque troncature**

In [ ]:
# Ex M28.6: the values you found on paper
answer_M28_6a = ...  # a) a list of 6 numbers (3 decimals)
answer_M28_6b = ...  # b) a list of 6 numbers (4 decimals)
answer_M28_6c = ...  # c) a list of 2 numbers (4 decimals): KL, then the half L1 norm
answer_M28_6d = ...  # d) a list of integers, by decreasing probability
answer_M28_6e = ...  # e) a list of 2 numbers (4 decimals): KL, then the half L1 norm
answer_M28_6f = ...  # f) an interval, as a text in French notation with decimal points, like "]2.5 ; 4]"

for letter, answer in zip("abcdef", [answer_M28_6a, answer_M28_6b, answer_M28_6c, answer_M28_6d, answer_M28_6e, answer_M28_6f]):
    wbm.check(f"M28.6{letter}", answer)

**Ex M28.7 — Les décimales de a/n et l'itération x ↦ x² + c modulo n : prépériode et période à la main, et la borne du principe des tiroirs**

In [ ]:
# Ex M28.7: the values you found on paper
answer_M28_7a = ...  # a) a list of 2 integers [mu, lambda]
answer_M28_7b = ...  # b) a list of 2 integers [mu, lambda]
answer_M28_7c = ...  # c) True or False

for letter, answer in zip("abc", [answer_M28_7a, answer_M28_7b, answer_M28_7c]):
    wbm.check(f"M28.7{letter}", answer)

**Ex M28.11 — Pont : diviser des scores par T avant la softmax, ou élever les probabilités à la puissance 1/T, la même loi ; et le même top-k**

In [ ]:
# Ex M28.11: the values you found on paper
answer_M28_11a = ...  # a) a list of 5 numbers (4 decimals)
answer_M28_11b = ...  # b) a list of 5 numbers (4 decimals)
answer_M28_11c = ...  # c) a list of 5 numbers (4 decimals)
answer_M28_11d = ...  # d) True or False
answer_M28_11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_M28_11a, answer_M28_11b, answer_M28_11c, answer_M28_11d, answer_M28_11e]):
    wbm.check(f"M28.11{letter}", answer)

## Partie A · Ta librairie : une carte à partir d'une matrice de Gram ; une loi tempérée et le plus petit ensemble de masse donnée

Fiche §2 et §5. Les trois fonctions de `mymath/sampling.py`, leurs tests, et un usage de chacune sur les données des exercices papier. La cellule de données ci-dessous recopie les données des exercices papier.

In [ ]:
# Tools of the notebook (parts A to D)
import math
import numbers
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, own_code, pair, run_pytest_group, verdict

TEST_M28 = "tests/test_m28_sampling.py"            # the tests of mymath/sampling.py
REF = wbm.load_mymath("ref", alias="mymath_ref")  # the reference: the checks compute their expected values with it
ref_sampling = REF.sampling
REF_SAMPLING = {f"ref_sampling.{name}": getattr(ref_sampling, name)    # a check stops if your code calls one of them
                for name in ("points_from_gram", "temper", "smallest_mass_set")}


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())[:160]
    except (TypeError, ValueError):
        return repr(value)[:160]


def check_answer(ex_id, value, hint=None):
    """wbm.check(ex_id, value, computed=True). For a list that holds True or False, the message gives only how many
    elements are right, not the place of the first wrong one (it would let you guess the booleans one by one).
    hint(value), when given, may return a targeted « Piste » for a wrong answer (None otherwise)."""
    holds_bool = isinstance(value, (list, tuple)) and any(isinstance(v, (bool, np.bool_)) for v in value)
    if not holds_bool and hint is None:
        return wbm.check(ex_id, value, computed=True)
    result = wbm.check(ex_id, value, computed=True, quiet=True)
    message = result.message
    if holds_bool:
        for cut in (" Le premier faux est le ", " Premier élément faux à l'indice "):
            message = message.split(cut)[0]
    if hint is not None and result.status not in ("correct", "pending", "unknown"):
        try:
            extra = hint(value)
        except Exception:  # noqa: BLE001 - an odd answer: no targeted hint
            extra = None
        if extra:
            message = f"{message} Piste : {extra}"
    icon = {"correct": "✅", "pending": "⏳", "unknown": "❓"}.get(result.status, "❌")
    print(f"{icon} Ex {ex_id} : {message}")
    return result


def check_rounding(ex_id, label, value, limit):
    """A number that must be a rounding error (0 <= value < limit): ✅, ⏳ or ❌."""
    if not filled(value):
        print(f"⏳ Ex {ex_id} : {label} : pas encore fait.")
        return None
    try:
        number = float(value)
    except (TypeError, ValueError):
        print(f"❌ Ex {ex_id} : {label} : attendu un nombre, reçu {value!r}.")
        return False
    ok = 0 <= number < limit
    print(f"✅ Ex {ex_id} : {label} : {number:.1e}, une erreur d'arrondi." if ok else
          f"❌ Ex {ex_id} : {label} : attendu un écart de l'ordre des arrondis (positif, moins de {limit:.0e}), reçu "
          f"{number!r}.")
    return ok


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def as_floats(value):
    """value as a float array, or None when it is not an array of numbers."""
    try:
        return np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None


def matches(got, want, rtol=1e-9, atol=1e-12):
    """True when got is an array of numbers of the shape of want and equal to it up to rounding (no error for an odd
    got)."""
    arr = as_floats(got)
    want = np.asarray(want, dtype=float)
    return arr is not None and arr.shape == want.shape and bool(np.allclose(arr, want, rtol=rtol, atol=atol))


def sq_dists_of(Z):
    """The squared distances between the rows of Z (to compare two maps: they ignore an isometry)."""
    Z = np.asarray(Z, dtype=float)
    return ((Z[:, None, :] - Z[None, :, :]) ** 2).sum(axis=-1)


_NUMBER = re.compile(r"[-−]?\d+(?:[.,]\d+)?(?:[eE][-−+]?\d+)?")


def numbers_in(text):
    """The numbers written in a text (a decimal point or comma, a minus sign − or -, an exponent e-07)."""
    found = []
    for token in _NUMBER.findall(text or ""):
        try:
            found.append(float(token.replace("−", "-").replace(",", ".")))
        except ValueError:
            pass
    return found


def gives(text, values, rel=0.01, abs_tol=0.006):
    """True when the text writes one of the values (to 1 % or to 0.006)."""
    return any(math.isclose(x, v, rel_tol=rel, abs_tol=abs_tol) for x in numbers_in(text) for v in values)


def line_points(line):
    """The (x, y) points of a line of matplotlib, as two float arrays."""
    return np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)


def drawn_points(ax, marker_only=False):
    """All the points drawn in ax: those of its lines (marker_only: lines with a marker only) and of its scatters."""
    points = []
    for line in ax.lines:
        if marker_only and line.get_marker() in (None, "", " ", "None", "none"):
            continue
        x, y = line_points(line)
        points += list(zip(x.ravel(), y.ravel()))
    for collection in ax.collections:
        offsets = np.asarray(collection.get_offsets(), dtype=float)
        if offsets.ndim == 2 and offsets.shape[1] == 2 and type(collection).__name__ == "PathCollection":
            points += [tuple(p) for p in offsets]
    return points


def points_at(ax, xs, ys, rtol=1e-7, atol=1e-12, marker_only=False):
    """True when every point (xs[k], ys[k]) is drawn in ax (a line with markers, or a scatter)."""
    points = np.array(drawn_points(ax, marker_only) or [(np.nan, np.nan)], dtype=float)
    return all(np.any(np.isclose(points[:, 0], x, rtol=rtol, atol=atol) & np.isclose(points[:, 1], y, rtol=rtol, atol=atol))
               for x, y in zip(np.atleast_1d(xs), np.atleast_1d(ys)))


def follows(ax, f, lo, hi, rtol=1e-6):
    """True when one line of ax, drawn with a stroke, has at least ten points with y = f(x), its x covering [lo, hi]."""
    for line in ax.lines:
        if str(line.get_linestyle()) in ("None", "none", "", " "):
            continue
        x, y = line_points(line)
        if (x.size >= 10 and np.all(np.isfinite(x)) and x.min() <= lo + 1e-9 * max(1.0, abs(lo))
                and x.max() >= hi - 1e-9 * max(1.0, abs(hi)) and np.allclose(y, f(x), rtol=rtol, atol=0)):
            return True
    return False


# The data of the paper exercises (02_exercices.md)
X_BEACONS = np.array([[3.0, 1.0], [1.0, 3.0], [-2.0, 0.0], [0.0, -2.0]])     # M28.1: the four beacons (m), in rows
D_RHOMBUS = np.array([[0.0, 29, 40, 29], [29, 0, 29, 42], [40, 29, 0, 29], [29, 42, 29, 0]])   # M28.2: A, B, C, D (km)
P_FORECAST = np.array([0.04, 0.19, 0.33, 0.26, 0.10, 0.08])                  # M28.6: the forecast, categories 0 to 5

### Ex M28.13 — Une configuration de points de matrice de Gram donnée, par le théorème spectral · `points_from_gram` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la construction de positions à partir d'une matrice de Gram, puis retrouver une carte à partir de distances, voir l'isométrie qui la sépare des vraies positions, et ce que deviennent des distances impossibles et une colonne de moins.  
**Prérequis :** fiche §2 ; M12c, M12d ; Ex M28.1 et M28.2 · **mymath :** `sampling.py` · **Parcours :** C

> **Mode d'emploi `mymath`** (comme en M27). Ouvre ta copie `mon_travail/mymath/sampling.py`, lis la docstring du module (ses conventions, et ce qui est permis) puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.sampling.<nom>` (par exemple `mymath.sampling.points_from_gram(G)`), ou donne-leur un nom court (`points_from_gram = mymath.sampling.points_from_gram`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute en tête du fichier `import numpy as np` (et `import math`, `import numbers` si tu t'en sers). NumPy est permis, mais de `np.linalg` seulement `eigh` (dans `points_from_gram`), et aucune fonction de SciPy ni de scikit-learn (`scipy.special.softmax` donnerait `temper`) : les tests comparent tes résultats à ceux de SciPy, et un test lit ton module. Une loi est donnée par ses **probabilités**, jamais par des scores ; rien n'est tiré au hasard ; ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy.

Écris `points_from_gram(G, dim=None, tol=1e-9)` (fiche §2), en suivant sa docstring : contrôle `G` (une matrice carrée non vide de nombres finis, symétrique au seuil $s = \texttt{tol}\cdot\max(1, \max\lvert G_{ij}\rvert)$ près), `dim` (un entier entre 1 et $n$, pas un booléen) et `tol` ($\ge 0$) ; diagonalise par `np.linalg.eigh`, qui range les valeurs propres par ordre **croissant** (retourne l'ordre) ; refuse une valeur propre sous $-s$ (une `ValueError` : $\mathbf{G}$ n'est pas une matrice de Gram) ; compte comme nulles celles de $[-s, s]$ ; garde `dim` colonnes, par défaut autant que de valeurs propres $> s$ (au moins une) ; multiplie chaque vecteur propre unitaire par la racine de sa valeur propre ; puis rends positive, dans chaque colonne, la première coordonnée de valeur absolue $> 10^{-8}$ ; rends un tableau de flottants de forme $(n, \texttt{dim})$.

**L'usage.** La cellule de données de la partie A donne les balises de M28.1 (`X_BEACONS`, en lignes) et les distances du losange de M28.2 (`D_RHOMBUS`, dans l'ordre A, B, C, D). Avec ta fonction :
- a) le losange : forme $\mathbf{G} = -\frac12\mathbf{J}\mathbf{D}^{(2)}\mathbf{J}$ à partir des **carrés** des distances, puis la carte (une matrice $4 \times 2$, telle que rendue), à comparer à ta réponse d) de M28.2 ;
- b) les balises : la carte $\mathbf{Y}$ rendue pour $\mathbf{G} = \mathbf{X}\mathbf{X}^\top$ (`Y_M28_13b`), le plus grand écart $\max\lvert\mathbf{Y}\mathbf{Y}^\top - \mathbf{G}\rvert$ (`gap_M28_13b`), la matrice $\mathbf{R} = \mathbf{Y}^\top\mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}$, de forme $(2, 2)$, telle que $\mathbf{y}_i = \mathbf{R}\mathbf{x}_i$ pour les quatre balises (`R_M28_13b`), puis son déterminant et vrai ou faux : $\mathbf{R}$ est orthogonale ($\mathbf{R}\mathbf{R}^\top = \mathbf{I}$ à $10^{-9}$ près) (une liste de 2 : le nombre, puis le booléen) ;
- c) le losange avec $\mathrm{AC} = 50$ km (M28.2 e)) : vrai ou faux, ta fonction lève une `ValueError`, puis la plus petite valeur propre de $-\frac12\mathbf{J}\mathbf{D}^{(2)}\mathbf{J}$, par `np.linalg.eigvalsh` (hors de `mymath`) (une liste de 2 : le booléen, puis le nombre) ;
- d) le losange avec `dim=1` : les quatre coordonnées de la carte à une colonne (`.ravel()`), puis $\lVert\mathbf{G} - \mathbf{X}\mathbf{X}^\top\rVert_F$ (une liste de 5 nombres).

Ce qu'il faut observer : la carte ne retrouve les vraies positions qu'à une isométrie près, ici une symétrie ($\det\mathbf{R} = -1$) ; des distances impossibles donnent une valeur propre négative, que la fonction refuse ; une colonne de moins, c'est la meilleure matrice de Gram de rang 1 (Eckart-Young), à une distance égale à la valeur propre laissée de côté.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M28")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M28, "test_points_from_gram_ or test_module_uses_")

**L'usage** : le losange, les balises, des distances impossibles, et une colonne de moins.

In [ ]:
answer_M28_13a = answer_M28_13b = answer_M28_13c = answer_M28_13d = ...
Y_M28_13b = gap_M28_13b = R_M28_13b = ...
with wbm.attempt("M28.13"):          # if you call points_from_gram here: ⏳ while it is not written
    answer_M28_13a = ...  # a) a (4, 2) array, as returned: the map of the rhombus, from G = -1/2 J D² J
    Y_M28_13b = ...       # b) the (4, 2) array returned for G = X Xᵀ (the beacons)
    gap_M28_13b = ...     # b) a number, as computed: the largest |Y Yᵀ − G|
    R_M28_13b = ...       # b) the (2, 2) array R = Yᵀ X (Xᵀ X)⁻¹
    answer_M28_13b = ...  # b) a list of 2: det R (as computed), then True or False: R Rᵀ = I (to 1e-9)
    answer_M28_13c = ...  # c) a list of 2: True or False: your function raises a ValueError for AC = 50 km, then the smallest eigenvalue of -1/2 J D² J (np.linalg.eigvalsh)
    answer_M28_13d = ...  # d) a list of 5 numbers, as computed: the 4 coordinates of the map with dim=1, then ‖G − X Xᵀ‖_F

In [ ]:
check_answer("M28.13a", answer_M28_13a)

with checked("M28.13"):                       # b): Y and R, compared with what the reference gives now
    results_13 = []
    report_13 = reporter(results_13, stop=False)
    G_c13 = X_BEACONS @ X_BEACONS.T
    want_Y_c13 = ref_sampling.points_from_gram(G_c13)        # computed by the reference at check time (no value written here)
    if not filled(Y_M28_13b):
        print("⏳ Ex M28.13b : la carte Y, pas encore faite.")
    else:
        Y_c13 = as_floats(Y_M28_13b)
        if Y_c13 is None or Y_c13.shape != want_Y_c13.shape:
            why_c13 = (f"Y a une ligne par balise et une colonne par valeur propre non nulle : attendu la forme "
                       f"{want_Y_c13.shape}, reçu {None if Y_c13 is None else Y_c13.shape}")
        elif matches(Y_c13, want_Y_c13, atol=1e-9):
            why_c13 = ""
        elif matches(Y_c13[:, ::-1], want_Y_c13, atol=1e-9):
            why_c13 = "les colonnes viennent par valeur propre décroissante (np.linalg.eigh les range dans l'ordre croissant)"
        elif matches(np.abs(Y_c13), np.abs(want_Y_c13), atol=1e-9):
            why_c13 = ("au signe près d'une colonne : dans chaque colonne, la première coordonnée de valeur absolue > 1e-8 "
                       "est positive (relance les tests de points_from_gram)")
        elif np.allclose(sq_dists_of(Y_c13), sq_dists_of(X_BEACONS), atol=1e-9):
            why_c13 = "Y a les bonnes distances, mais pas la carte que donne la construction de la docstring"
        else:
            why_c13 = "Y Yᵀ doit redonner G = X Xᵀ (relance les tests de points_from_gram)"
        report_13(not why_c13, "Ex M28.13b : la carte Y est celle de la construction de la docstring",
                  f"Ex M28.13b : la carte Y : {why_c13}")
    check_rounding("M28.13b", "le plus grand écart |Y Yᵀ − G|", gap_M28_13b, 1e-12)
    if not filled(R_M28_13b):
        print("⏳ Ex M28.13b : la matrice R, pas encore faite.")
    else:
        Y_c13 = as_floats(Y_M28_13b) if filled(Y_M28_13b) else None
        own_c13 = Y_c13 is not None and Y_c13.shape == (4, 2) and np.allclose(Y_c13 @ Y_c13.T, G_c13, atol=1e-9)
        base_c13 = Y_c13 if own_c13 else want_Y_c13          # R is judged with your map Y when Y Yᵀ = G
        want_R_c13 = base_c13.T @ X_BEACONS @ np.linalg.inv(X_BEACONS.T @ X_BEACONS)   # the formula of the statement
        R_c13 = as_floats(R_M28_13b)
        if matches(R_c13, want_R_c13, atol=1e-9):
            why_c13 = ""
        elif matches(R_c13, want_R_c13.T, atol=1e-9):
            why_c13 = "y_i = R x_i pour les points en lignes s'écrit Y = X Rᵀ : R = Yᵀ X (Xᵀ X)⁻¹, pas sa transposée"
        else:
            why_c13 = "R = Yᵀ X (Xᵀ X)⁻¹, de forme (2, 2), avec ta carte Y"
        report_13(not why_c13, "Ex M28.13b : R envoie chaque balise sur sa ligne de Y", f"Ex M28.13b : R : {why_c13}")
    check_answer("M28.13b", answer_M28_13b)

for letter, answer in zip("cd", [answer_M28_13c, answer_M28_13d]):
    check_answer(f"M28.13{letter}", answer)

### Ex M28.14 — Tempérer une loi donnée par ses probabilités, et le plus petit ensemble de masse donnée · `temper`, `smallest_mass_set` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer la température sur des probabilités et le plus petit ensemble de masse donnée, puis voir ce que la température fait à l'ensemble d'un top-p et à l'entropie.  
**Prérequis :** fiche §5 ; M17 ; Ex M28.6 · **mymath :** `sampling.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M28.13 : écris les fonctions dans ta copie `mon_travail/mymath/sampling.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `temper(p, temperature)` et `smallest_mass_set(p, mass)` (fiche §5), en suivant leurs docstrings. `temper` : contrôle la loi (1-D, non vide, des nombres finis $\ge 0$ de somme 1 à $10^{-9}$ près) et la température ($\ge 0$, `np.inf` permis, ni `nan` ni booléen) ; traite d'abord $T = 0$ (la loi uniforme sur les issues les plus probables) et $T = \infty$ (la loi uniforme sur les issues de probabilité non nulle) ; sinon, calcule par les logarithmes des seuls $p_i > 0$, moins leur maximum, divisés par $T$, puis l'exponentielle et la division par la somme, et laisse les zéros à 0. `smallest_mass_set` : contrôle la loi et la masse ($0 < \texttt{mass} \le 1$) ; range les issues par probabilité décroissante avec un tri **stable** (`np.argsort(-p, kind="stable")` : à égalité, la plus petite issue d'abord) ; cumule jusqu'à ce que le total atteigne `mass` $- 10^{-9}$ ; rends les indices gardés, dans cet ordre, en tableau d'entiers.

**L'usage.** La cellule de données de la partie A donne la prévision de M28.6 (`P_FORECAST`). Avec tes fonctions :
- a) `temper(P_FORECAST, 1/3)`, puis `smallest_mass_set(P_FORECAST, 0.87)`, à comparer à tes réponses a) et d) de M28.6 (une liste de 10 nombres : les 6 probabilités, puis les 4 indices) ;
- b) pour $\mathbf{q} = (0{,}3 ;\ 0 ;\ 0{,}3 ;\ 0{,}25 ;\ 0{,}15)$ : `temper(q, 0)`, puis `temper(q, np.inf)` (une liste de 10 nombres) ;
- c) les tailles de l'ensemble du top-p de seuil 0,87 de la prévision tempérée, pour $T = \frac13$, 1, 3 et $\infty$, dans cet ordre (une liste de 4 entiers) ;
- d) les entropies, en nats, de la prévision tempérée à $T = \frac13$, 1 et 3 (calculées hors de `mymath`, sans compter les issues de probabilité nulle), puis vrai ou faux : elles croissent (une liste de 4 : les trois nombres, puis le booléen).

Ce qu'il faut observer : refroidir rétrécit l'ensemble du top-p, réchauffer l'agrandit ; à $T = \infty$, les zéros restent nuls ; l'entropie croît avec $T$ vers $\ln 6$.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M28")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M28, "test_temper_ or test_smallest_mass_set_")

**L'usage** : la prévision refroidie et son top-p, les deux limites, les tailles du top-p, et l'entropie.

In [ ]:
Q_14 = np.array([0.3, 0.0, 0.3, 0.25, 0.15])
answer_M28_14a = answer_M28_14b = answer_M28_14c = answer_M28_14d = ...
with wbm.attempt("M28.14"):          # if you call temper or smallest_mass_set here: ⏳ while they are not written
    answer_M28_14a = ...  # a) a list of 10 numbers: temper(P_FORECAST, 1/3) (6 probabilities), then smallest_mass_set(P_FORECAST, 0.87) (4 indices)
    answer_M28_14b = ...  # b) a list of 10 numbers: temper(Q_14, 0), then temper(Q_14, np.inf)
    answer_M28_14c = ...  # c) a list of 4 integers: the sizes of the top-p set of mass 0.87 of the tempered forecast, for T = 1/3, 1, 3, np.inf
    answer_M28_14d = ...  # d) a list of 4: the entropies (nats) of the tempered forecast at T = 1/3, 1, 3, then True or False: they grow

In [ ]:
for letter, answer in zip("abcd", [answer_M28_14a, answer_M28_14b, answer_M28_14c, answer_M28_14d]):
    check_answer(f"M28.14{letter}", answer)

## Partie B · Prédire, dessiner

Fiche §3 et §4. Ce qu'un changement d'unité fait à un pas de montée, prédit puis vérifié ; des motifs rayés, leur flou et leur lissage par la variation totale, dessinés.

### Ex M28.15 — Prédire ce que deviennent un pas η∇f et un pas normalisé quand la fonction est multipliée par 1 000, puis vérifier 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir ce qu'un changement d'unité fait à un pas de montée ordinaire et à un pas normalisé, avec une projection sur un pavé, puis le vérifier.  
**Prérequis :** fiche §3 ; M05b, M19 ; Ex M28.3 · **Parcours :** C, E

Deux réglages $\mathbf{x} \in [0, 1]^2$ commandent un capteur, dont le signal $f(\mathbf{x}) = -\lVert\mathbf{x} - \mathbf{c}\rVert^2$, en watts, est maximal en $\mathbf{c} = (0{,}33 ;\ 0{,}58)$ (`C_15`), à l'intérieur du pavé. On fait 20 pas de montée projetée, $\mathbf{x}_{t+1} = \operatorname{clip}(\mathbf{x}_t + \eta\,\mathbf{s}_t,\ 0,\ 1)$, de pas $\eta = 0{,}08$ (`ETA_15`), depuis $\mathbf{x}_0 = (0{,}83 ;\ 0{,}13)$ (`X0_15`), avec le pas ordinaire $\mathbf{s}_t = \nabla f(\mathbf{x}_t)$ ou le pas normalisé $\mathbf{s}_t = \nabla f(\mathbf{x}_t)/\lVert\nabla f(\mathbf{x}_t)\rVert$. Puis on exprime le signal en milliwatts : la fonction devient $1\,000f$, et l'on recommence avec le même $\eta$. **Avant d'exécuter quoi que ce soit**, prédis :
- a) avec $1\,000f$ et le pas ordinaire, les itérés : (A) vont au maximum, plus vite qu'avec $f$ ; (B) s'arrêtent sur un sommet du pavé et y restent ; (C) sautent d'un sommet à un autre sans jamais s'arrêter ; (D) sortent du pavé (une lettre) ;
- b) le point $\mathbf{x}_1$ avec $1\,000f$ et le pas ordinaire (une liste de 2 nombres, que tu calcules à la main) ;
- c) avec le pas normalisé, les itérés de $1\,000f$ : (A) sont ceux de $f$ ; (B) font des pas 1 000 fois plus longs (une lettre) ;
- d) avec le pas normalisé, le premier pas $k$ où $\lVert\mathbf{x}_k - \mathbf{c}\rVert < 0{,}1$ (un entier).

Puis, une fois tes prédictions écrites et vérifiées, écris `ascent_15(scale, normalized, n_steps=20)`, qui rend le tableau des itérés, de forme $(\texttt{n\_steps} + 1, 2)$, ligne 0 le départ, pour la fonction $\texttt{scale}\cdot f$ ; elle lit `C_15`, `X0_15` et `ETA_15` au moment de l'appel (la vérification change `X0_15` un instant : un départ caché). Relève :
- e) les distances $\lVert\mathbf{x}_{20} - \mathbf{c}\rVert$ des quatre montées, dans l'ordre ($f$, pas ordinaire), ($1\,000f$, pas ordinaire), ($f$, pas normalisé), ($1\,000f$, pas normalisé) (une liste de 4 nombres ; 4 décimales dans ta copie) ;
- f) vrai ou faux : avec le pas normalisé, les 21 itérés de $f$ et de $1\,000f$ sont les mêmes.

La vérification de l'expérience ne s'exécute qu'une fois tes quatre prédictions écrites. Ce qu'il faut observer : multiplier la fonction par 1 000 multiplie le pas ordinaire par 1 000, et la projection transforme ces sauts démesurés en allers-retours entre deux sommets, alors que le maximum est à l'intérieur ; le pas normalisé ne voit pas l'unité, mais, de longueur fixe, il oscille près du maximum sans s'y poser.

In [ ]:
C_15 = np.array([0.33, 0.58])      # the maximum of f(x) = −‖x − c‖² (W), inside the box [0, 1]²
X0_15 = np.array([0.83, 0.13])     # the start
ETA_15 = 0.08                      # the step

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M28_15a = ...  # a) "A", "B", "C" or "D", written BEFORE the experiment: 1000 f with the plain step, the iterates...
prediction_M28_15b = ...  # b) a list of 2 numbers, computed by hand: x1 for 1000 f with the plain step
prediction_M28_15c = ...  # c) "A" or "B": the normalized iterates of 1000 f
prediction_M28_15d = ...  # d) an integer: with the normalized step, the first k such that ‖x_k − c‖ < 0.1

In [ ]:
for letter, prediction in zip("abcd", [prediction_M28_15a, prediction_M28_15b, prediction_M28_15c, prediction_M28_15d]):
    wbm.check(f"M28.15{letter}", prediction)

**L'expérience.** Écris-la une fois tes quatre prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
def ascent_15(scale, normalized, n_steps=20):
    """The iterates of the projected ascent x <- clip(x + ETA_15 s, 0, 1) on scale·f, f(x) = −‖x − C_15‖², from X0_15
    (read at the call), with s the gradient (normalized=False) or the gradient divided by its norm (normalized=True): an
    array of shape (n_steps + 1, 2), row 0 the start."""
    raise NotImplementedError("ascent_15 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M28_15e = answer_M28_15f = ...
with wbm.attempt("M28.15"):          # your function: ⏳ while it is not written
    answer_M28_15e = ...  # e) a list of 4 numbers, as computed: ‖x20 − c‖ for (f, plain), (1000 f, plain), (f, normalized), (1000 f, normalized)
    answer_M28_15f = ...  # f) True or False: with the normalized step, the 21 iterates of f and of 1000 f are the same

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M28_15a, prediction_M28_15b, prediction_M28_15c, prediction_M28_15d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    for letter, answer in zip("ef", [answer_M28_15e, answer_M28_15f]):
        check_answer(f"M28.15{letter}", answer)

    HIDDEN_C15 = [([0.12, 0.94], 1.0, False, 6,
  [[0.12, 0.94], [0.15360000000000001, 0.8824], [0.181824, 0.834016], [0.20553216000000002, 0.79337344],
   [0.22544701440000003, 0.7592336896], [0.24217549209600003, 0.730556299264],
   [0.25622741336064003, 0.70646729138176]]),
 ([0.12, 0.94], 1.0, True, 6,
  [[0.12, 0.94], [0.1603096820419269, 0.8708976879281253], [0.20061936408385378, 0.8017953758562506],
   [0.24092904612578067, 0.7326930637843759], [0.28123872816770756, 0.6635907517125013],
   [0.3215484102096345, 0.5944884396406266], [0.3618580922515614, 0.5253861275687519]]),
 ([0.12, 0.94], 3.0, False, 6,
  [[0.12, 0.94], [0.2208, 0.7671999999999999], [0.273216, 0.677344], [0.30047232, 0.63061888],
   [0.3146456064, 0.6063218175999999], [0.322015715328, 0.593687345152],
   [0.32584817197056, 0.58711741947904]]),
 ([0.95, 0.05], 12.0, False, 6,
  [[0.95, 0.05], [0.0, 1.0], [0.6336, 0.19359999999999988], [0.050687999999999955, 0.9354880000000001],
   [0.5869670400000001, 0.2529510399999999], [0.09359032319999994, 0.8808850432],
   [0.547496902656, 0.30318576025599986]])]   # hidden calls (start, scale, normalized, n_steps) and the iterates expected

    with checked("M28.15"):
        results_15 = []
        report_15 = reporter(results_15, stop=False)
        start_c15 = np.array(X0_15, dtype=float)
        got_c15 = []
        try:
            for x0_c15, scale_c15, normalized_c15, n_c15, _ in HIDDEN_C15:
                X0_15 = np.array(x0_c15)                     # your function reads X0_15 at the call
                got_c15.append(ascent_15(scale_c15, normalized_c15, n_c15))
        finally:
            X0_15 = start_c15
        for (x0_c15, scale_c15, normalized_c15, n_c15, want_c15), result_c15 in zip(HIDDEN_C15, got_c15):
            arr_c15 = as_floats(result_c15)
            label_c15 = f"un départ caché, scale = {scale_c15:g}, pas {'normalisé' if normalized_c15 else 'ordinaire'}"
            if arr_c15 is None or arr_c15.shape != (n_c15 + 1, 2):
                why_c15 = (f"attendu un tableau de forme ({n_c15 + 1}, 2), ligne 0 le départ, reçu "
                           f"{None if arr_c15 is None else arr_c15.shape}")
            elif matches(arr_c15, want_c15, atol=1e-9):
                why_c15 = ""
            elif matches(arr_c15[0], start_c15):
                why_c15 = "ta fonction part encore du départ de l'énoncé : lis X0_15 au moment de l'appel"
            elif np.any(arr_c15 < -1e-12) or np.any(arr_c15 > 1 + 1e-12):
                why_c15 = "les itérés doivent rester dans le pavé : projette chaque point (np.clip) après le pas"
            else:
                why_c15 = ("ces itérés ne sont pas ceux de la montée projetée de l'énoncé, x ← clip(x + η s, 0, 1), sur scale·f "
                           "(le gradient de f est −2(x − c))")
            report_15(not why_c15, f"ascent_15 : {label_c15}, comme attendu", f"ascent_15, {label_c15} : {why_c15}")
        verdict("M28.15", all(results_15), "ta montée suit l'énoncé depuis un départ caché aussi.",
                "relis les lignes ❌ ci-dessus.")

**Après l'expérience**, dans ta copie : pourquoi les itérés de $1\,000f$ ne s'arrêtent-ils jamais ? Pourquoi le pas normalisé ne se pose-t-il pas sur le maximum, et que faudrait-il changer pour qu'il s'en approche autant qu'on veut ?

### Ex M28.16 — Des motifs rayés de fréquence croissante jusqu'au damier : leur variation totale, et ce qu'en laissent un flou 3 × 3 et dix pas de descente sur la variation totale 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner des motifs de plus en plus rapides, leur flou et leur lissage par la variation totale, et lire sur les images et sur une courbe ce que coûte une haute fréquence.  
**Prérequis :** fiche §4 ; M21 ; Ex M28.4 et M28.5 · **Parcours :** C, E

Sur une grille de $48 \times 48$ cases à **bords libres** (une case du bord a moins de quatre voisines), cinq motifs (`MOTIFS_16`, un dictionnaire, de A à E) : A $\cos\frac{2\pi j}{16}$, B $\cos\frac{2\pi j}{6}$, C $\cos\frac{2\pi j}{3}$, D $(-1)^j$ et E $(-1)^{i+j}$, le damier. Le notebook fournit `blur3_16(x)`, la moyenne des neuf cases autour de chaque case intérieure (un tableau $46 \times 46$).

Écris `grid_laplacian_16(x)`, qui rend $\mathbf{L}\mathbf{x}$ pour la grille à bords libres (en chaque case, son nombre de voisines fois sa valeur, moins la somme de ses voisines ; sans boucle sur les cases), puis `draw_stripes_16(axes, motifs, eta, n_steps)`, qui reçoit une grille d'axes de forme $(5, 3)$ et un dictionnaire de motifs et, pour chaque motif (une ligne d'axes), dessine (`imshow`, une même échelle de couleurs de $-1$ à 1 pour les trois images : `vmin=-1, vmax=1`) le motif, son flou, et la grille après `n_steps` pas de descente $\mathbf{x} \leftarrow \mathbf{x} - \eta\cdot 2\mathbf{L}\mathbf{x}$ ; chaque image a un titre qui donne la valeur mesurée (le rapport, le gain, l'atténuation). La fonction rend un dictionnaire de trois listes, dans l'ordre des motifs : `"ratio"`, $\mathrm{VT}(\mathbf{x})/\lVert\mathbf{x}\rVert^2$ sur toute la grille (avec $\mathrm{VT}(\mathbf{x}) = \mathbf{x}^\top\mathbf{L}\mathbf{x}$) ; `"gain"`, $\langle\text{flou}, \mathbf{x}_{\text{int}}\rangle/\lVert\mathbf{x}_{\text{int}}\rVert^2$, où $\mathbf{x}_{\text{int}}$ est le motif sur les cases intérieures ; `"decay"`, $\lVert\mathbf{y}_c\rVert/\lVert\mathbf{x}_c\rVert$ après les pas de descente, sur le bloc central `[12:36, 12:36]`. Écris aussi `draw_decay_curve_16(ax, eta, n_steps, freqs, decays)`, qui trace la courbe $\big(1 - 8\eta\sin^2(\pi f)\big)^{n}$ pour $f$ de 0 à $\frac12$, et place les points mesurés des rayures A à D (`freqs` $= (\frac{1}{16}, \frac16, \frac13, \frac12)$, `FREQS_16`), avec une échelle logarithmique en ordonnée, des noms d'axes, et en titre la valeur théorique du damier, $(1 - 16\eta)^{n}$. Les deux fonctions dessinent dans les axes reçus, sans ouvrir de figure.

La vérification essaie `grid_laplacian_16` sur des grilles tirées au hasard, appelle les deux fonctions avec $\eta = 0{,}05$ et 10 pas (`ETA_16`, `STEPS_16`), affiche les figures, et compare tes trois listes aux valeurs attendues ; elle les appelle aussi sur d'autres motifs, un autre pas et un autre nombre de pas (des appels cachés). Relève, avec `draw_stripes_16` (appelle-la sur les axes d'une figure que tu fermes aussitôt, `fig, axes = plt.subplots(5, 3)` puis `plt.close(fig)`, pour ne pas afficher la figure deux fois) :
- a) les cinq rapports (une liste de 5 nombres ; 3 décimales dans ta copie) ;
- b) les cinq gains (une liste de 5 nombres ; 4 décimales dans ta copie) ;
- c) le logarithme décimal des cinq atténuations (une liste de 5 nombres ; 2 décimales dans ta copie) ;
- d) vrai ou faux : les rapports mesurés sont un peu sous les valeurs $4\sin^2(\pi f)$ (et 8 pour le damier) de la grille refermée ;
- e) vrai ou faux : sur le bloc central, les atténuations sont exactement celles de la courbe pour A à D, et la valeur du titre pour le damier.

Ce qu'il faut observer : plus le motif est rapide, plus il coûte en variation totale et plus le flou et la descente l'effacent ; le flou rend la période 3 nulle et le damier divisé par 9 ; sur une grille à bords libres, les cases du bord ont moins de voisines, ce qui fait baisser les rapports ; mais un pas ne mélange que des voisines, et, après 10 pas, une case à plus de 10 cases du bord n'a pas encore « vu » le bord : au centre, la courbe est exacte.

In [ ]:
N_16 = 48
I_16, J_16 = np.meshgrid(np.arange(N_16), np.arange(N_16), indexing="ij")    # row i, column j
MOTIFS_16 = {"A": np.cos(2 * np.pi * J_16 / 16), "B": np.cos(2 * np.pi * J_16 / 6), "C": np.cos(2 * np.pi * J_16 / 3),
             "D": (-1.0) ** J_16, "E": (-1.0) ** (I_16 + J_16)}          # five 48 × 48 motifs, from A to E
FREQS_16 = [1 / 16, 1 / 6, 1 / 3, 1 / 2]                                  # the frequencies of A to D (cycles per cell)
ETA_16, STEPS_16 = 0.05, 10


def blur3_16(x):
    """The mean of the nine cells around each inner cell of the grid x: an array of shape (n − 2, m − 2)."""
    x = np.asarray(x, dtype=float)
    n, m = x.shape
    return sum(x[1 + a:n - 1 + a, 1 + b:m - 1 + b] for a in (-1, 0, 1) for b in (-1, 0, 1)) / 9

In [ ]:
def grid_laplacian_16(x):
    """L x for the grid x with free edges: in each cell, its number of neighbours times its value, minus the sum of its
    neighbours (no loop over the cells). Returns a new array of the shape of x."""
    raise NotImplementedError("grid_laplacian_16 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_stripes_16(axes, motifs, eta, n_steps):
    """axes: a (5, 3) grid of axes; motifs: a dict {name: grid}. For each motif (one row of axes): imshow of the motif,
    of its blur (blur3_16) and of the grid after n_steps descent steps x <- x - eta·2Lx, with vmin=-1, vmax=1, and a
    title that gives the value measured. Returns {"ratio": [...], "gain": [...], "decay": [...]} in the order of the
    motifs: xᵀLx/‖x‖², ⟨blur, x_int⟩/‖x_int‖², ‖y_c‖/‖x_c‖ on the block [12:36, 12:36]. No other axes."""
    raise NotImplementedError("draw_stripes_16 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_decay_curve_16(ax, eta, n_steps, freqs, decays):
    """In ax: the curve (1 − 8 eta sin²(πf))**n_steps for f from 0 to 1/2, the points (freqs, decays) with a marker, a
    log scale on the y axis, axis labels, and as title the theoretical value of the checkerboard, (1 − 16 eta)**n_steps.
    No other axes."""
    raise NotImplementedError("draw_decay_curve_16 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M28_16a = answer_M28_16b = answer_M28_16c = answer_M28_16d = answer_M28_16e = ...
with wbm.attempt("M28.16"):          # your functions: ⏳ while they are not written
    answer_M28_16a = ...  # a) a list of 5 numbers, as computed: the ratios VT/‖x‖² of A to E
    answer_M28_16b = ...  # b) a list of 5 numbers, as computed: the gains of the 3 × 3 blur
    answer_M28_16c = ...  # c) a list of 5 numbers, as computed: log10 of the decays on the central block
    answer_M28_16d = ...  # d) True or False: the ratios measured are a little below 4 sin²(πf) (and 8 for the checkerboard)
    answer_M28_16e = ...  # e) True or False: on the central block, the decays are exactly those of the curve for A to D, and the value of the title for E

In [ ]:
for letter, answer in zip("abcde", [answer_M28_16a, answer_M28_16b, answer_M28_16c, answer_M28_16d, answer_M28_16e]):
    check_answer(f"M28.16{letter}", answer)

HIDDEN_C16 = {'eta': 0.04,
 'n_steps': 6,
 'ratio': [0.5822120027257839, 1.7269281881521568, 1.9993937387495189, 3.9166666666666665, 2.0906812371280066],
 'gain': [0.804737854124365, 0.4706742197221989, 0.33333333333333337, -0.3333333333333333, 0.40396548564918133],
 'decay': [0.7497771577520788, 0.40143149043579374, 0.35129803161599993, 0.09886748262400001,
           0.3250567416365988]}   # a hidden call: eta, n_steps, and the three lists expected for five hidden motifs (below)
HIDDEN_MOTIFS_C16 = {"H1": np.cos(2 * np.pi * I_16 / 8), "H2": np.cos(2 * np.pi * (I_16 + 2 * J_16) / 10),
                     "H3": np.cos(2 * np.pi * J_16 / 4 + 0.7), "H4": (-1.0) ** I_16,
                     "H5": np.cos(2 * np.pi * (I_16 / 5 - J_16 / 7))}


def tv_c16(x):
    """VT(x) by its definition (fiche §4): the squares of the differences between vertical and horizontal neighbours."""
    return float((np.diff(x, axis=0) ** 2).sum() + (np.diff(x, axis=1) ** 2).sum())


def laplacian_why_c16(rng):
    """'' when grid_laplacian_16 behaves as a laplacian with free edges on random grids, else the reason."""
    for shape in ((5, 7), (6, 4)):
        x, y = rng.normal(size=shape), rng.normal(size=shape)
        x_before = x.copy()
        Lx, Ly, Lxy = (as_floats(grid_laplacian_16(v)) for v in (x, y, x + 2 * y))
        if any(v is None or v.shape != shape for v in (Lx, Ly, Lxy)):
            return f"grid_laplacian_16 doit rendre un tableau de la forme de x, {shape}"
        if not np.array_equal(x, x_before):
            return "grid_laplacian_16 modifie la grille reçue : travaille sur une copie, ou crée un nouveau tableau"
        if not np.allclose(Lxy, Lx + 2 * Ly, atol=1e-9):
            return "L est linéaire : L(x + 2y) doit valoir Lx + 2Ly"
        if math.isclose(float(x.ravel() @ Lx.ravel()), -tv_c16(x), rel_tol=1e-9):
            return "(Lx)_a = deg(a) x_a − la somme des voisines de a, pas l'inverse"
        if abs(float(Lx.sum())) > 1e-9:
            return ("la somme des coefficients de Lx est nulle : en chaque case, son nombre de voisines (2 dans un coin, 3 "
                    "sur un bord, 4 à l'intérieur) fois sa valeur, moins la somme de ses voisines")
        if not math.isclose(float(x.ravel() @ Lx.ravel()), tv_c16(x), rel_tol=1e-9):
            return ("xᵀLx doit valoir VT(x), la somme des carrés des différences entre voisines de la grille : à bords "
                    "libres, la dernière colonne n'est pas voisine de la première (np.roll referme la grille)")
        if not math.isclose(float(y.ravel() @ Lx.ravel()), float(x.ravel() @ Ly.ravel()), rel_tol=1e-9, abs_tol=1e-12):
            return "L est symétrique : ⟨Lx, y⟩ doit valoir ⟨x, Ly⟩"
    return ""


def stripes_why_c16(axes, motifs, result, want=None, letter_check=True):
    """What is wrong in a call of draw_stripes_16 (French phrases): its three lists, then the fifteen images."""
    problems = []
    if not isinstance(result, dict) or set(result) != {"ratio", "gain", "decay"}:
        return [f"draw_stripes_16 doit rendre un dictionnaire de clés \"ratio\", \"gain\", \"decay\", reçu {shown(result)}"]
    lists = {key: as_floats(result[key]) for key in ("ratio", "gain", "decay")}
    if any(v is None or v.shape != (len(motifs),) for v in lists.values()):
        return [f"chaque liste a une valeur par motif, {len(motifs)} en tout, dans l'ordre des motifs"]
    if want is None:                     # the visible call: compared like the answers a) to c) (answers.json)
        for key, letter in (("ratio", "a"), ("gain", "b")):
            if not wbm.check(f"M28.16{letter}", lists[key].tolist(), computed=True, quiet=True).passed:
                problems.append(f"la liste \"{key}\" n'est pas celle qu'attend l'énoncé")
        if np.any(lists["decay"] <= 0) or not wbm.check("M28.16c", np.log10(lists["decay"]).tolist(), computed=True,
                                                        quiet=True).passed:
            problems.append("la liste \"decay\" n'est pas celle qu'attend l'énoncé (‖y_c‖/‖x_c‖ sur le bloc central "
                            "[12:36, 12:36], après les pas de descente)")
    else:
        for key in ("ratio", "gain", "decay"):
            if not np.allclose(lists[key], want[key], rtol=1e-6, atol=1e-9):
                problems.append(f"la liste \"{key}\" n'est pas celle qu'attend l'énoncé pour ces motifs, ce pas et ce "
                                "nombre de pas (dessine les motifs reçus, avec eta et n_steps reçus)")
    for k, (name, x) in enumerate(motifs.items()):
        row = []
        for j in range(3):
            images = axes[k, j].images
            if not images:
                row.append(f"l'image {j + 1} (imshow)")
                continue
            data = np.asarray(images[-1].get_array(), dtype=float)
            if images[-1].get_clim() != (-1.0, 1.0):
                row.append(f"l'échelle de couleurs de −1 à 1 de l'image {j + 1} (vmin=-1, vmax=1)")
            if j == 0 and not matches(data, x):
                row.append("le motif lui-même en première image")
            if j == 1 and not matches(data, blur3_16(x)):
                row.append("son flou (blur3_16) en deuxième image")
            if j == 2 and (data.shape != x.shape or not math.isclose(
                    float(np.linalg.norm(data[12:36, 12:36]) / np.linalg.norm(x[12:36, 12:36])), float(lists["decay"][k]),
                    rel_tol=1e-6, abs_tol=1e-12)):
                row.append("la grille après les pas de descente en troisième image (celle dont \"decay\" est mesuré)")
            value = [lists["ratio"][k], lists["gain"][k], lists["decay"][k]][j]
            candidates = [value] + ([math.log10(value)] if j == 2 and value > 0 else [])
            if not gives(axes[k, j].get_title(), candidates):
                row.append(f"un titre qui donne la valeur mesurée sur l'image {j + 1} (un nombre)")
        if row:
            problems.append(f"motif {name} : il manque " + " ; ".join(row))
    return problems


def curve_why_c16(ax, eta, n_steps, freqs, decays):
    """What is missing in a drawing of draw_decay_curve_16 (French phrases)."""
    missing = []
    if not follows(ax, lambda f: (1 - 8 * eta * np.sin(np.pi * f) ** 2) ** n_steps, 0.0, 0.5):
        missing.append("la courbe (1 − 8η sin²(πf))ⁿ, tracée d'un trait pour f de 0 à 1/2 avec le η et le n reçus")
    if not points_at(ax, freqs, decays, marker_only=True):
        missing.append("les points mesurés (freqs, decays), avec un marqueur")
    if ax.get_yscale() != "log":
        missing.append("l'échelle logarithmique en ordonnée (ax.set_yscale(\"log\"))")
    if not ax.get_xlabel() or not ax.get_ylabel():
        missing.append("des noms d'axes")
    if not gives(ax.get_title(), [(1 - 16 * eta) ** n_steps], rel=0.01, abs_tol=0.0):
        missing.append("en titre, la valeur théorique du damier, (1 − 16η)ⁿ")
    return missing


with checked("M28.16"):
    results_16 = []
    report_16 = reporter(results_16, stop=False)
    why_c16 = laplacian_why_c16(np.random.default_rng())          # random grids, drawn at each run
    report_16(not why_c16, "grid_laplacian_16 : linéaire, symétrique, de somme nulle, et xᵀLx = VT(x) sur deux grilles "
              "tirées au hasard", f"grid_laplacian_16 : {why_c16}")
    fig_16, axes_16 = plt.subplots(5, 3, figsize=(9, 15))
    fig_curve_16, ax_curve_16 = plt.subplots(figsize=(6.5, 4.2))
    try:
        result_16 = []
        opened_16 = drawn(lambda: result_16.append(draw_stripes_16(axes_16, MOTIFS_16, ETA_16, STEPS_16)))
        problems_16 = stripes_why_c16(axes_16, MOTIFS_16, result_16[0])
        report_16(not problems_16, "draw_stripes_16 (η = 0,05, 10 pas) : les trois listes comme attendu, et les quinze "
                  "images, leur échelle et leurs titres", "draw_stripes_16 : " + " ; ".join(problems_16))
        fig_c16, axes_c16 = plt.subplots(5, 3)
        try:
            hidden_c16 = []
            opened_16 += drawn(lambda: hidden_c16.append(draw_stripes_16(axes_c16, HIDDEN_MOTIFS_C16, HIDDEN_C16["eta"],
                                                                          HIDDEN_C16["n_steps"])))
            problems_c16 = stripes_why_c16(axes_c16, HIDDEN_MOTIFS_C16, hidden_c16[0], want=HIDDEN_C16)
        finally:
            plt.close(fig_c16)
        report_16(not problems_c16, "un appel caché de draw_stripes_16 (cinq autres motifs, un autre pas, un autre nombre "
                  "de pas) est juste aussi", "un appel caché de draw_stripes_16 : " + " ; ".join(problems_c16))
        decays_16 = as_floats(result_16[0].get("decay")) if isinstance(result_16[0], dict) else None
        decays_16 = decays_16[:4] if decays_16 is not None and decays_16.shape == (5,) else np.array([0.8, 0.3, 0.03, 0.006])
        opened_16 += drawn(draw_decay_curve_16, ax_curve_16, ETA_16, STEPS_16, FREQS_16, decays_16)
        missing_16 = curve_why_c16(ax_curve_16, ETA_16, STEPS_16, FREQS_16, decays_16)
        report_16(not missing_16, "draw_decay_curve_16 : la courbe, les points, l'échelle logarithmique, les noms d'axes et "
                  "le titre", "draw_decay_curve_16 : il manque " + " ; ".join(missing_16))
        fig_c16, ax_c16 = plt.subplots()
        try:
            opened_16 += drawn(draw_decay_curve_16, ax_c16, 0.03, 8, [0.1, 0.25, 0.4], [0.8, 0.45, 0.15])
            missing_c16 = curve_why_c16(ax_c16, 0.03, 8, [0.1, 0.25, 0.4], [0.8, 0.45, 0.15])
        finally:
            plt.close(fig_c16)
        report_16(not missing_c16, "un appel caché de draw_decay_curve_16 (η = 0,03, 8 pas, trois autres points) est juste "
                  "aussi", "un appel caché de draw_decay_curve_16 (η = 0,03, 8 pas) : il manque " + " ; ".join(missing_c16))
        report_16(not opened_16, "tes deux fonctions dessinent dans les axes reçus",
                  "tes fonctions doivent dessiner dans les axes reçus, sans ouvrir de figure")
    except BaseException:
        plt.close(fig_16)
        plt.close(fig_curve_16)
        raise
    fig_16.tight_layout()
    fig_curve_16.tight_layout()
    plt.show()
    verdict("M28.16", all(results_16), "ton laplacien, tes quinze images et ta courbe sont justes.",
            "relis les lignes ❌ ci-dessus.")

**Après les figures**, dans ta copie : pourquoi le flou efface-t-il exactement C et pas D ? Pourquoi les rapports de la grille à bords libres sont-ils un peu sous ceux de la grille refermée ? Que deviendrait l'atténuation au centre avec 30 pas ?

## Partie C · Déboguer, tester

Fiche §2, §3 et §6. Trois erreurs classiques, qui passent une vérification mal choisie ; un test de propriété de `points_from_gram`. Les versions fausses et le code du collègue s'appuient sur la référence (`ref_sampling`), pour ne pas dépendre de ton code.

### Ex M28.17 — Chasse à l'erreur : un double centrage fait sur les distances au lieu de leurs carrés, une montée qui coupe le gradient au lieu du point, une prépériode comptée dans la période 🐛 ★ ⏱️ 20 min
**Objectif :** reconnaître trois erreurs classiques sur le double centrage, la montée projetée et la période, qui passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §2, §3 et §6 ; Ex M28.2, M28.3 et M28.7 · **Parcours :** C, E

Un collègue a écrit trois fonctions, chacune accompagnée d'une vérification qui passe (cellule fournie) :
1. `map_from_distances(D, dim=2)` doit rendre une carte (une ligne par lieu) dont les distances sont celles de la matrice `D` ; elle forme $\mathbf{J}$, calcule `G = -0.5 * J @ D @ J`, puis rend `points_from_gram(G, dim)`. Il l'a vérifiée sur un triangle équilatéral de côté 1 : les trois distances de la carte valent 1.
2. `tune_emitters(x0, grad, eta=0.05, n_steps=120, lower=0.0, upper=1.0)` doit faire `n_steps` pas de montée projetée sur le pavé $[\texttt{lower}, \texttt{upper}]^n$, le gradient étant donné par la fonction `grad` ; elle répète `x = x + eta * np.clip(grad(x), lower, upper)`. Il l'a vérifiée sur $f(\mathbf{x}) = -\lVert\mathbf{x} - \mathbf{c}\rVert^2$ avec $\mathbf{c} = (0{,}45 ;\ 0{,}62 ;\ 0{,}88)$, depuis $(0{,}15 ;\ 0{,}25 ;\ 0{,}05)$, avec les réglages par défaut : la montée arrive en $\mathbf{c}$, à $10^{-5}$ près, à l'intérieur du pavé $[0, 1]^3$.
3. `cycle_of(f, x0)` doit rendre `(mu, lam)`, la prépériode et la période de la suite $u_{t+1} = f(u_t)$ ; elle note dans un dictionnaire l'indice où chaque valeur apparaît, et, à la première valeur déjà vue, d'indice $j$, rend l'indice $i$ de sa première apparition et `j`. Il l'a vérifiée sur $x \mapsto (x + 3) \bmod 10$ depuis 0 : `(0, 10)`.

À faire : trouve et corrige chaque erreur ; les fonctions corrigées s'appellent `map_from_distances_fixed`, `tune_emitters_fixed` et `cycle_of_fixed`, avec les mêmes arguments. Puis réponds :
- a) pour le losange de M28.2 (`D_RHOMBUS`), les distances AB, AC et BD de la carte de `map_from_distances_fixed`, puis de celle de `map_from_distances` (une liste de 6 nombres) ;
- b) pour quatre émetteurs, $f(\mathbf{x}) = -\lVert\mathbf{x} - \mathbf{c}\rVert^2$ avec $\mathbf{c} = (1{,}3 ;\ -0{,}2 ;\ 0{,}6 ;\ 0{,}35)$, depuis $\mathbf{x}_0 = (0{,}2 ;\ 0{,}7 ;\ 0{,}78 ;\ 0{,}1)$, avec les réglages par défaut : le réglage final de `tune_emitters_fixed`, puis celui de `tune_emitters` (une liste de 8 nombres) ;
- c) pour le générateur jouet $x \mapsto (x^2 + 3) \bmod 22$, depuis 2 : `cycle_of_fixed`, puis `cycle_of` (une liste de 4 entiers).

Dans ta copie : pour chacune des trois erreurs, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie tes corrections sur des cas cachés : des nuages de points qui ne sont pas des triangles équilatéraux, des montées sur d'autres pavés dont le maximum n'est pas à l'intérieur, des suites avec une prépériode.

In [ ]:
# The colleague's code
def map_from_distances(D, dim=2):
    """A map (one row per place) whose distances are those of the matrix D."""
    D = np.asarray(D, dtype=float)
    n = len(D)
    J = np.eye(n) - np.ones((n, n)) / n
    G = -0.5 * J @ D @ J
    return ref_sampling.points_from_gram(G, dim)


def tune_emitters(x0, grad, eta=0.05, n_steps=120, lower=0.0, upper=1.0):
    """n_steps steps of projected gradient ascent on the box [lower, upper]^n; grad(x) gives the gradient."""
    x = np.array(x0, dtype=float)
    for _ in range(n_steps):
        x = x + eta * np.clip(grad(x), lower, upper)
    return x


def cycle_of(f, x0):
    """(mu, lam): the preperiod and the period of the sequence u_(t+1) = f(u_t), from u_0 = x0."""
    seen, x, t = {}, x0, 0
    while x not in seen:
        seen[x] = t
        x, t = f(x), t + 1
    return seen[x], t


# The colleague's checks: all pass
triangle_17 = np.array([[0.0, 1, 1], [1, 0, 1], [1, 1, 0]])
map_check_17 = map_from_distances(triangle_17)
print("(1)", np.round(np.sqrt(((map_check_17[:, None] - map_check_17[None]) ** 2).sum(-1)), 6).tolist(),
      "(attendu des 1 hors de la diagonale)")
c_check_17 = np.array([0.45, 0.62, 0.88])
x_check_17 = tune_emitters(np.array([0.15, 0.25, 0.05]), lambda x: -2.0 * (x - c_check_17))
print("(2)", bool(np.allclose(x_check_17, c_check_17, atol=1e-5)), bool(np.all((0 <= x_check_17) & (x_check_17 <= 1))),
      "(attendu True, True)")
print("(3)", cycle_of(lambda x: (x + 3) % 10, 0), "(attendu (0, 10))")

In [ ]:
def map_from_distances_fixed(D, dim=2):
    """A map (one row per place) whose distances are those of the matrix D."""
    raise NotImplementedError("map_from_distances_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def tune_emitters_fixed(x0, grad, eta=0.05, n_steps=120, lower=0.0, upper=1.0):
    """n_steps steps of projected gradient ascent on the box [lower, upper]^n; grad(x) gives the gradient."""
    raise NotImplementedError("tune_emitters_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def cycle_of_fixed(f, x0):
    """(mu, lam): the preperiod and the period of the sequence u_(t+1) = f(u_t), from u_0 = x0."""
    raise NotImplementedError("cycle_of_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M28_17a = answer_M28_17b = answer_M28_17c = ...
with wbm.attempt("M28.17"):          # your three functions: ⏳ while they are not written
    answer_M28_17a = ...  # a) a list of 6 numbers, as computed: AB, AC, BD of the map of map_from_distances_fixed(D_RHOMBUS), then of the WRONG map_from_distances(D_RHOMBUS)
    answer_M28_17b = ...  # b) a list of 8 numbers, as computed: the final setting of tune_emitters_fixed, then of the WRONG tune_emitters
    answer_M28_17c = ...  # c) a list of 4 integers: cycle_of_fixed, then the WRONG cycle_of, for x -> (x² + 3) mod 22 from 2

In [ ]:
for letter, answer in zip("abc", [answer_M28_17a, answer_M28_17b, answer_M28_17c]):
    check_answer(f"M28.17{letter}", answer)

CASES_C17 = {'maps': [{'points': [[-9.57, -0.74], [-7.27, 8.52], [8.8, -0.6], [7.53, -9.0], [-3.34, -7.2], [6.82, 7.82],
                      [8.68, 5.17]],
           'dim': 2},
          {'points': [[-2.62, 1.46, -2.5], [-1.73, 1.73, 5.01], [5.18, 2.97, 2.69], [3.58, -0.07, -3.4],
                      [5.72, 1.11, -4.3], [-1.18, -1.74, 2.08]],
           'dim': 3}],
 'emitters': [{'A': [[2.0, 0.8, 0.0], [0.8, 1.5, 0.5], [0.0, 0.5, 1.0]],
               'c': [1.6, -0.9, 0.7],
               'x0': [0.1, 0.4, 0.9],
               'eta': 0.05,
               'n_steps': 400,
               'lower': -0.5,
               'upper': 1.2},
              {'A': [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
               'c': [-0.3, 1.25, 0.45, 0.8],
               'x0': [0.8, 0.1, 0.9, 0.2],
               'eta': 0.05,
               'n_steps': 200,
               'lower': 0.0,
               'upper': 1.0}],
 'cycles': [{'map': 'squares', 'x0': 3, 'mu': 5, 'lam': 2}, {'map': 'affine', 'x0': 2, 'mu': 3, 'lam': 5},
            {'map': 'automaton', 'x0': (1, 3), 'mu': 6, 'lam': 4}]}   # the hidden cases of the three corrected functions
CYCLES_C17 = {"squares": lambda x: (x * x + 5) % 26, "affine": lambda x: (6 * x + 1) % 40,
              "automaton": lambda s: (s[1], (s[0] * s[1] + 2) % 7)}       # the maps of the hidden sequences

results_17, parts_17 = [], []


def report_17(ok, success, failure):
    results_17.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_17(part, name, function, *args, **kwargs):
    """function(*args, **kwargs); an error of the learner's code becomes one ❌ line that names the function (the check of
    this part stops there, the final verdict stays)."""
    try:
        return function(*args, **kwargs)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        hint = ""
        if part == 1 and isinstance(exc, ValueError) and "semi-definite" in str(exc):
            hint = (" ; −½ J D J, fait sur les distances, n'est en général pas une matrice de Gram : le double centrage "
                    "se fait sur leurs carrés")
        report_17(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:160]}{hint}")
        parts_17.append(part)
        raise StopCheck from None


with checked("M28.17"):                                       # (1) map_from_distances_fixed
    wrong_17 = []
    for case_c17 in CASES_C17["maps"]:
        points_c17 = np.array(case_c17["points"])
        D_c17 = np.sqrt(sq_dists_of(points_c17))
        before_c17 = D_c17.copy()
        got_c17 = call_17(1, "map_from_distances_fixed", map_from_distances_fixed, D_c17, case_c17["dim"])
        arr_c17 = as_floats(got_c17)
        shape_c17 = (len(points_c17), case_c17["dim"])
        if arr_c17 is None or arr_c17.shape != shape_c17:
            wrong_17.append(f"attendu une carte de forme {shape_c17} (une ligne par lieu, dim colonnes), reçu "
                            f"{None if arr_c17 is None else arr_c17.shape}")
            continue
        if not np.array_equal(D_c17, before_c17):
            wrong_17.append("ta fonction modifie la matrice D reçue : travaille sur une copie")
            continue
        d_c17, tol_c17 = np.sqrt(sq_dists_of(arr_c17)), 1e-7 * D_c17.max()
        if np.allclose(d_c17, D_c17, rtol=1e-7, atol=tol_c17):
            continue
        if np.allclose(d_c17, math.sqrt(2) * D_c17, rtol=1e-7, atol=tol_c17):
            wrong_17.append("G = −½ J D⁽²⁾ J : sans le facteur −½, chaque distance est multipliée par √2")
        else:
            wrong_17.append(f"{len(points_c17)} lieux dans un espace de dimension {case_c17['dim']} : les distances de ta "
                            "carte ne sont pas celles de D (le double centrage se fait sur les carrés des distances, "
                            "coefficient par coefficient)")
    report_17(not wrong_17, "(1) map_from_distances_fixed : deux nuages cachés, dans le plan et dans l'espace",
              "(1) map_from_distances_fixed : " + " ; ".join(list(dict.fromkeys(wrong_17))[:2]))
    parts_17.append(1)

with checked("M28.17"):                                       # (2) tune_emitters_fixed
    wrong_17 = []
    for case_c17 in CASES_C17["emitters"]:
        A_c17, c_c17 = np.array(case_c17["A"]), np.array(case_c17["c"])
        lower_c17, upper_c17 = case_c17["lower"], case_c17["upper"]

        def grad_c17(x, A=A_c17, c=c_c17):
            return -2.0 * A @ (np.asarray(x, dtype=float) - c)

        got_c17 = call_17(2, "tune_emitters_fixed", tune_emitters_fixed, np.array(case_c17["x0"]), grad_c17,
                          eta=case_c17["eta"], n_steps=case_c17["n_steps"], lower=lower_c17, upper=upper_c17)
        x_c17 = as_floats(got_c17)
        if x_c17 is None or x_c17.shape != c_c17.shape:
            wrong_17.append(f"attendu le réglage final, un tableau de forme {c_c17.shape}, reçu {shown(got_c17)}")
            continue
        if np.any(x_c17 < lower_c17 - 1e-12) or np.any(x_c17 > upper_c17 + 1e-12):
            wrong_17.append(f"le réglage final sort du pavé [{fr(lower_c17, 1)} ; {fr(upper_c17, 1)}] : projette le point "
                            "(np.clip) après chaque pas, avec les bornes lower et upper reçues")
            continue
        g_c17 = grad_c17(x_c17)
        at_low, at_up = x_c17 <= lower_c17 + 1e-9, x_c17 >= upper_c17 - 1e-9     # the optimality conditions on a box
        if np.all(np.abs(g_c17[~at_low & ~at_up]) <= 1e-6) and np.all(g_c17[at_low] <= 1e-6) and np.all(g_c17[at_up] >= -1e-6):
            continue
        if np.allclose(x_c17, np.clip(c_c17, lower_c17, upper_c17), atol=1e-6):
            wrong_17.append("projeter seulement à la fin donne la projection du maximum sans contrainte, qui n'est pas le "
                            "maximum sur le pavé quand l'énergie a des termes croisés : projette le point après chaque pas")
        elif np.any((g_c17 < -1e-6) & ~at_low):
            wrong_17.append("une coordonnée dont le gradient est négatif reste au-dessus de sa borne : elle doit pouvoir "
                            "baisser (on ne coupe pas le gradient, on projette le point)")
        else:
            wrong_17.append("la montée projetée s'arrête là où aucun pas permis ne fait plus monter f, ce qui n'est pas le "
                            "cas ici : projette le point après chaque pas, pas seulement à la fin")
    report_17(not wrong_17, "(2) tune_emitters_fixed : deux montées cachées, sur d'autres pavés, dont une énergie à "
              "termes croisés", "(2) tune_emitters_fixed : " + " ; ".join(list(dict.fromkeys(wrong_17))[:2]))
    parts_17.append(2)

with checked("M28.17"):                                       # (3) cycle_of_fixed
    wrong_17 = []
    for case_c17 in CASES_C17["cycles"]:
        got_c17 = call_17(3, "cycle_of_fixed", cycle_of_fixed, CYCLES_C17[case_c17["map"]], case_c17["x0"])
        try:
            mu_c17, lam_c17 = (int(v) for v in got_c17)
        except (TypeError, ValueError):
            wrong_17.append(f"cycle_of_fixed doit rendre le couple (mu, lam), reçu {shown(got_c17)}")
            continue
        if (mu_c17, lam_c17) == (case_c17["mu"], case_c17["lam"]):
            continue
        if (mu_c17, lam_c17) == (case_c17["mu"], case_c17["mu"] + case_c17["lam"]):
            wrong_17.append("tu rends encore l'indice de la première répétition : la période est la longueur du cycle, "
                            "sans la queue")
        elif (mu_c17, lam_c17) == (case_c17["lam"], case_c17["mu"]):
            wrong_17.append("cycle_of_fixed rend (mu, lam) : la prépériode, puis la période")
        else:
            wrong_17.append("une suite cachée avec une prépériode : ton couple (mu, lam) n'est pas le bon (mu est l'indice "
                            "de la première apparition de la valeur répétée)")
    report_17(not wrong_17, "(3) cycle_of_fixed : trois suites cachées avec une prépériode, dont une sur des couples",
              "(3) cycle_of_fixed : " + " ; ".join(list(dict.fromkeys(wrong_17))[:2]))
    parts_17.append(3)

if len(set(parts_17)) == 3:
    verdict("M28.17", all(results_17), "les trois fonctions corrigées passent tous les cas.", "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque erreur, pourquoi la vérification du collègue ne la voyait-elle pas, et quelle vérification l'aurait vue ?

### Ex M28.18 — Un test de propriété : la configuration rendue redonne la matrice de Gram et les distances, pour des points tirés au hasard 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test de propriété à tolérance relative, qui compte ses échecs sur des nuages tirés au hasard, et voir ce qu'il attrape et ce qu'il ne peut pas attraper.  
**Prérequis :** fiche §2 ; Ex M28.13 · **Parcours :** C

Écris `gram_property_test_18(n_trials, rng, func=None, rtol=1e-9)` ; quand `func` vaut `None`, le test prend `mymath.sampling.points_from_gram` **au moment de l'appel**. Pour chacun des `n_trials` essais, dans cet ordre : `n = int(rng.integers(2, 9))`, `d = int(rng.integers(1, 5))`, `scale = 10.0 ** rng.uniform(-2, 3)`, `P = scale * rng.standard_normal((n, d))`, puis, si `rng.random() < 0.3`, `P[-1] = P[0]` (un point répété), puis `t = scale * rng.standard_normal(d)` ; ensuite `G = P @ P.T`, $s = \max(1, \max\lvert G_{ij}\rvert)$ et `X = func(G)`. La fonction rend un dictionnaire de quatre compteurs d'essais (un essai compte une fois s'il échoue) :
- `"shape"` : `X` n'a pas la forme $(n, r)$, avec $r$ le rang de `P` (`np.linalg.matrix_rank`), ou contient un nombre non fini ;
- `"gram"` : $\max\lvert\mathbf{X}\mathbf{X}^\top - \mathbf{G}\rvert > \texttt{rtol}\cdot s$ (ou `X` non fini) ;
- `"distances"` : les carrés des distances des lignes de `X` s'écartent de ceux de `P` de plus de $4\,\texttt{rtol}\cdot s$ (ou `X` non fini) ;
- `"translation"` : la carte `func(Gc)` de la matrice doublement centrée `Gc` $= -\frac12\mathbf{J}\mathbf{D}^{(2)}\mathbf{J}$ des points translatés `P + t` a des carrés de distances qui s'écartent de ceux de `P` de plus de $4\,\texttt{rtol}\cdot\max(1, \max\lvert(\mathbf{G}_c)_{ij}\rvert)$ (ou n'est pas finie).

Les carrés des distances se calculent dans le test, avec NumPy, sans appeler `mymath`. La cellule fournie donne cinq versions fausses de `points_from_gram` : `no_sqrt` (les colonnes $\lambda_k\mathbf{q}_k$), `ascending` (l'ordre croissant de `eigh`, non retourné), `all_columns` ($n$ colonnes, sans mettre à 0 les petites valeurs propres : la racine d'une valeur propre de $-10^{-15}$ donne `nan`), `rotated` (la bonne carte, tournée) et `no_sign` (la bonne carte, sans la convention de signe). Puis :
- a) pourquoi une tolérance relative à $s$ (dans ta copie) ? Combien de fois sur 300 essais une tolérance absolue de $10^{-9}$ sur $\max\lvert\mathbf{X}\mathbf{X}^\top - \mathbf{G}\rvert$ refuserait-elle la bonne fonction (mêmes tirages, avec `np.random.default_rng(6618)`) (un entier) ?
- b) avec `rng = np.random.default_rng(6618)` et 300 essais, avec ta fonction : vrai ou faux, les quatre compteurs sont nuls ;
- c) avec le même départ (un nouveau `default_rng(6618)` chaque fois, 300 essais) et chacune des cinq versions fausses, dans l'ordre `no_sqrt`, `ascending`, `all_columns`, `rotated`, `no_sign` : les quatre compteurs, dans l'ordre `"shape"`, `"gram"`, `"distances"`, `"translation"` (une liste de 5 listes de 4 entiers) ;
- d) les noms des versions fausses qui passent le test (leurs quatre compteurs nuls) (une liste de noms, dans n'importe quel ordre, comme `["no_sqrt"]`) ; dans ta copie : pourquoi c'est normal, et qu'est-ce qui, dans le dépôt, vérifie ce qu'elles ont de faux ? Pourquoi `ascending` n'est-elle attrapée que par une partie des essais ?

La vérification lance ton test sur deux départs cachés (40 essais chacun) : avec la référence, où ses quatre compteurs doivent être nuls, puis avec quatre versions fausses cachées, écrites dans la cellule de vérification, qui faussent à chaque essai la forme, la matrice de Gram seule (une carte déplacée), la translation, ou les produits scalaires et les distances d'un point (son opposé) ; elle le lance aussi avec `rtol = 1e-6` sur une version fausse de $10^{-7}$ en valeur relative (ton test doit se servir de `rtol`), et remplace un instant `mymath.sampling.points_from_gram` par une version fausse qui ajoute une colonne : tes compteurs doivent alors compter. Chacune des quatre propriétés est ainsi exigée par au moins une version fausse. Un entier NumPy convient pour un compteur ; un test qui échoue ne lève pas d'`AssertionError` : il compte. Ce qu'il faut retenir : ce test ajoute à ceux de `tests/test_m28_sampling.py` des points répétés et une translation suivie d'un double centrage ; il attrape trois versions fausses, et ne peut pas voir une isométrie, puisque la matrice de Gram et les distances n'en dépendent pas ; un test se juge aux erreurs qu'il attrape, et à celles qu'il ne peut pas attraper.

In [ ]:
def no_sqrt(G, dim=None, tol=1e-9):
    """WRONG: the columns λ_k q_k (the square root forgotten), built on the reference."""
    X = ref_sampling.points_from_gram(G, dim, tol)
    return X * np.sqrt((X ** 2).sum(axis=0))


def ascending(G, dim=None, tol=1e-9):
    """WRONG: the increasing order of np.linalg.eigh, not reversed."""
    G = np.asarray(G, dtype=float)
    s = tol * max(1.0, float(np.abs(G).max()))
    w, V = np.linalg.eigh(G)
    w = np.where(w > s, w, 0.0)
    r = max(1, int(np.count_nonzero(w))) if dim is None else dim
    return V[:, :r] * np.sqrt(w[:r])


def all_columns(G, dim=None, tol=1e-9):
    """WRONG: n columns, the small eigenvalues not set to 0 (the square root of −1e-15 gives nan)."""
    w, V = np.linalg.eigh(np.asarray(G, dtype=float))
    w, V = w[::-1], V[:, ::-1]
    with np.errstate(invalid="ignore"):
        return V * np.sqrt(w)


def rotated(G, dim=None, tol=1e-9):
    """WRONG: the right map, turned."""
    X = ref_sampling.points_from_gram(G, dim, tol)
    k = X.shape[1]
    Q, _ = np.linalg.qr(np.arange(1.0, k * k + 1).reshape(k, k) + 3 * np.eye(k))
    return X @ Q


def no_sign(G, dim=None, tol=1e-9):
    """WRONG: the right map, without the sign convention."""
    return -ref_sampling.points_from_gram(G, dim, tol)

In [ ]:
def gram_property_test_18(n_trials, rng, func=None, rtol=1e-9):
    """n_trials random clouds: n = int(rng.integers(2, 9)), d = int(rng.integers(1, 5)), scale = 10.0 ** rng.uniform(-2, 3),
    P = scale * rng.standard_normal((n, d)), P[-1] = P[0] if rng.random() < 0.3, t = scale * rng.standard_normal(d);
    func defaults to mymath.sampling.points_from_gram, looked up at the call. Returns the numbers of trials "shape",
    "gram", "distances", "translation" that fail."""
    raise NotImplementedError("gram_property_test_18 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M28_18a = answer_M28_18b = answer_M28_18c = answer_M28_18d = ...
with wbm.attempt("M28.18"):          # your test and your library: ⏳ while they are not written
    answer_M28_18a = ...  # a) an integer: the trials (default_rng(6618), 300 trials) where max|X Xᵀ − G| > 1e-9 for the right function
    answer_M28_18b = ...  # b) True or False: the four counters are 0 (default_rng(6618), 300 trials, your function)
    answer_M28_18c = ...  # c) a list of 5 lists of 4 integers: the counters ("shape", "gram", "distances", "translation") for no_sqrt, ascending, all_columns, rotated, no_sign (a new default_rng(6618) each time, 300 trials)
    answer_M28_18d = ...  # d) a list of names (strings, any order): the wrong versions whose four counters are 0

In [ ]:
for letter, answer in zip("abcd", [answer_M28_18a, answer_M28_18b, answer_M28_18c, answer_M28_18d]):
    check_answer(f"M28.18{letter}", answer)

def extra_column_c18(G, dim=None, tol=1e-9):
    """WRONG: the right map and a column of zeros more (only its shape is wrong)."""
    X = ref_sampling.points_from_gram(G, dim, tol)
    return np.hstack([X, np.zeros((len(X), 1))])


def shifted_c18(G, dim=None, tol=1e-9):
    """WRONG: the right map moved by a same vector (its distances stay right)."""
    X = ref_sampling.points_from_gram(G, dim, tol)
    return X + 0.01 * math.sqrt(max(1.0, float(np.abs(G).max())))


def ramp_c18(G, dim=None, tol=1e-9):
    """WRONG: the right map and a column (0, c, 2c, ...) more."""
    X = ref_sampling.points_from_gram(G, dim, tol)
    c = 0.01 * math.sqrt(max(1.0, float(np.abs(G).max())))
    return np.hstack([X, c * np.arange(len(X), dtype=float)[:, None]])


def centered_c18(G, dim=None, tol=1e-9):
    """WRONG only for a doubly centered matrix (rows of sum 0): then the column (0, c, 2c, ...) more."""
    G = np.asarray(G, dtype=float)
    if float(np.abs(G.sum(axis=1)).max()) <= 1e-9 * max(1.0, float(np.abs(G).max())):
        return ramp_c18(G, dim, tol)
    return ref_sampling.points_from_gram(G, dim, tol)


def flipped_row_c18(G, dim=None, tol=1e-9):
    """WRONG for a matrix that is not doubly centered: the first point replaced by its opposite (its norm is kept, its
    dot products and its distances to the others are not)."""
    G = np.asarray(G, dtype=float)
    X = ref_sampling.points_from_gram(G, dim, tol)
    if float(np.abs(G.sum(axis=1)).max()) > 1e-9 * max(1.0, float(np.abs(G).max())):
        X = X.copy()
        X[0] = -X[0]
    return X


def scaled_c18(G, dim=None, tol=1e-9):
    """WRONG by 1e-7 in relative value: seen with rtol = 1e-9 on most trials, never with rtol = 1e-6."""
    return (1 + 1e-7) * ref_sampling.points_from_gram(G, dim, tol)


with checked("M28.18"):
    results_18 = []
    report_18 = reporter(results_18, stop=False)
    N_C18 = 40                                      # trials per hidden start
    KEYS_C18 = ("shape", "gram", "distances", "translation")
    PROPERTIES_C18 = {"shape": "la forme (n, r) de X, r le rang de P, et ses nombres finis",
                      "gram": "que X Xᵀ redonne G à rtol·s près",
                      "distances": "que les carrés des distances des lignes de X redonnent ceux de P",
                      "translation": "la carte de la matrice doublement centrée des points translatés P + t"}

    def all_c18(*keys):
        """The counters expected from a version built to break exactly these properties at every trial."""
        return {key: (N_C18 if key in keys else 0) for key in KEYS_C18}

    def run_c18(label, rng, **kwargs):
        """Your gram_property_test_18 on a hidden start: its dictionary."""
        try:
            counts = gram_property_test_18(N_C18, rng, **kwargs)
        except (NotImplementedError, MymathMissingError):
            raise
        except AssertionError as exc:
            print(f"❌ Ex M28.18 : gram_property_test_18 ({label}) lève AssertionError ({str(exc)[:120]}) : elle doit "
                  "compter les essais qui échouent et rendre le dictionnaire, sans assert.")
            raise StopCheck from None
        except Exception as exc:  # noqa: BLE001 - the learner's error
            print(f"❌ Ex M28.18 : gram_property_test_18 ({label}) lève {type(exc).__name__} : {str(exc)[:200]}")
            raise StopCheck from None
        if not isinstance(counts, dict) or set(counts) != set(KEYS_C18):
            print("❌ Ex M28.18 : gram_property_test_18 doit rendre un dictionnaire de clés " + ", ".join(KEYS_C18)
                  + f", reçu {shown(counts)[:160]}")
            raise StopCheck
        for key in KEYS_C18:
            if isinstance(counts[key], (bool, np.bool_)) or not isinstance(counts[key], (int, np.integer)):
                print(f"❌ Ex M28.18 : le compteur \"{key}\" doit être un nombre entier d'essais (un entier NumPy "
                      f"convient), reçu {type(counts[key]).__name__} {counts[key]!r}")
                raise StopCheck
        return {key: int(v) for key, v in counts.items()}

    def compare_c18(label, got, want):
        """One ✅ when the four counters are those expected; else one ❌ per counter, naming the property."""
        wrong = [k for k in KEYS_C18 if got[k] != want[k]]
        if not wrong:
            report_18(True, f"{label} : les quatre compteurs comme attendu", "")
        for k in wrong:
            if got[k] == 0:
                why = f"ton test ne vérifie pas {PROPERTIES_C18[k]}"
            elif want[k] == 0:
                why = ("cette version ne fausse pas cette propriété : ton contrôle compte trop (sa tolérance relative "
                       "rtol·s, ce qu'il compare)")
            else:
                why = (f"cette version fausse {PROPERTIES_C18[k]} à chaque essai : ton test n'en compte qu'une partie "
                       "(un essai compte une fois s'il échoue)")
            report_18(False, "", f"{label} : \"{k}\" = {got[k]}, attendu {want[k]} : {why}")

    for start_c18, seed_c18 in enumerate((6641, 6643)):          # the two hidden starts
        for label_c18, kwargs_c18, want_c18 in (
                ("la référence", {"func": ref_sampling.points_from_gram}, all_c18()),
                ("une version fausse cachée n° 1", {"func": extra_column_c18}, all_c18("shape")),
                ("une version fausse cachée n° 2", {"func": shifted_c18}, all_c18("gram")),
                ("une version fausse cachée n° 3", {"func": centered_c18}, all_c18("translation")),
                ("une version fausse cachée n° 4", {"func": flipped_row_c18}, all_c18("gram", "distances"))):
            label_c18 = f"{label_c18}, départ caché {start_c18 + 1}"
            compare_c18(label_c18, run_c18(label_c18, np.random.default_rng(seed_c18), **kwargs_c18), want_c18)
    seed_c18 = (6641, 6643)[0]
    compare_c18("une version fausse de 1e-7, avec rtol = 1e-6 (ton test doit se servir de rtol)",
                run_c18("rtol = 1e-6", np.random.default_rng(seed_c18), func=scaled_c18, rtol=1e-6), all_c18())

    original_c18 = mymath.sampling.points_from_gram
    aliases_c18 = [alias for alias, value in list(globals().items())
                   if value is original_c18 and alias != "original_c18"]     # your own names for it
    calls_c18 = [0]

    def replaced_c18(G, dim=None, tol=1e-9):
        calls_c18[0] += 1
        return ramp_c18(G, dim, tol)

    mymath.sampling.points_from_gram = replaced_c18
    for alias_c18 in aliases_c18:
        globals()[alias_c18] = replaced_c18
    try:
        patched_c18 = run_c18("une version fausse à la place de ta points_from_gram", np.random.default_rng(seed_c18))
    finally:
        mymath.sampling.points_from_gram = original_c18
        for alias_c18 in aliases_c18:
            globals()[alias_c18] = original_c18
    if calls_c18[0] == 0:
        report_18(False, "", "une version fausse remplace un instant ta points_from_gram, mais ton test ne l'appelle jamais : "
                  "avec func=None, il doit prendre mymath.sampling.points_from_gram au moment de l'appel, sans en garder "
                  "une copie")
    else:
        compare_c18("une version fausse à la place de ta points_from_gram (func=None)", patched_c18,
                    all_c18(*KEYS_C18))
    verdict("M28.18", all(results_18), "ton test compte juste sur la référence et sur les versions fausses cachées, se "
            "sert de rtol, et prend ta fonction au moment de l'appel.", "relis les lignes ❌ ci-dessus.")

**Dans ta copie** : a) pourquoi une tolérance relative à $s$ ; d) pourquoi deux versions fausses passent, ce qui dans le dépôt vérifie ce qu'elles ont de faux, et pourquoi `ascending` n'est attrapée que par une partie des essais.

**Après la vérification**, dans ta copie : quelle propriété faudrait-il ajouter pour attraper une carte tournée, et pourquoi n'est-ce plus un test de la matrice de Gram ?

## Partie D · Défi

Fiche §2. Une carte retrouvée à partir de distances incomplètes et bruitées, jugée après alignement sur quatre régions.

### Ex M28.19 — Défi : redessiner la carte de 30 villes quand un quart des distances manquent, à moins de 1 km en moyenne après alignement 🏆 ★★★ ⏱️ 30 min
**Objectif :** retrouver une carte à partir de distances incomplètes et bruitées, en itérant le positionnement multidimensionnel classique, et la juger après alignement.  
**Prérequis :** fiche §2 ; M12d ; Ex M28.2, M28.13 et M28.17 · **Parcours :** C

La cellule fournie donne la matrice `D_19`, de forme $(30, 30)$, des distances entre 30 villes d'une région de $200 \times 150$ km : symétrique, de diagonale nulle, mesurée avec une erreur d'environ 300 m (des écarts gaussiens d'écart-type 0,3 km), et dont un quart des 435 paires, tirées au hasard, manquent (`np.nan`) ; elle est tirée par `towns_19(6619)`, qui rend aussi les vraies positions `P_19`. Elle donne aussi `procrustes_error_19(Y, P)`, qui centre la carte `Y` et les vraies positions `P`, trouve la transformation orthogonale qui aligne au mieux `Y` sur `P` (le problème de Procrustes orthogonal, Borg et Groenen, ch. 19 *Procrustes Procedures*) et rend la distance moyenne, en km, entre les villes alignées et leurs vraies positions.

Écris `map_from_partial_distances_19(D)`, qui reçoit une telle matrice ($n$ villes) et rend une carte de forme $(n, 2)$. Tout NumPy et SciPy sont permis, et ta `points_from_gram` ; la matrice doublement centrée d'une matrice complétée a en général des valeurs propres négatives, que ta `points_from_gram` refuse : remplace-la d'abord par sa projection sur les matrices semi-définies positives (ses valeurs propres négatives mises à 0, par `np.linalg.eigh`).

**Critère de réussite :** une erreur après alignement de moins de 1 km en moyenne pour la région du notebook, et pour trois régions cachées de la vérification (d'autres villes, de 26 à 34, d'autres paires manquantes, et des villes renumérotées, tournées et déplacées à chaque exécution), en moins de 10 secondes par région. La vérification recalcule aussi ton erreur a) avec ta fonction, et refuse une fonction qui rend la même carte pour deux régions. Relève :
- a) ton erreur après alignement pour la région du notebook (un nombre, tel que calculé, en km ; 3 décimales dans ta copie) ;
- b) l'erreur qu'on obtient en remplaçant chaque distance manquante par la moyenne des distances connues, puis en faisant un seul positionnement multidimensionnel classique (un nombre, tel que calculé, en km ; 1 décimale dans ta copie).

Point de départ : un seul double centrage d'une matrice complétée au hasard ne suffit pas, mais une carte donne des distances pour toutes les paires.

In [ ]:
from scipy.linalg import orthogonal_procrustes
from scipy.spatial.distance import pdist, squareform


def towns_19(seed, n=30, frac=0.25, noise=0.3):
    """n towns drawn in a 200 × 150 km region, their distances measured with gaussian errors of standard deviation noise
    (km), and a fraction frac of the pairs, drawn at random, missing (np.nan). Returns (P, D): the true positions, of
    shape (n, 2), and the (n, n) matrix of the measured distances (symmetric, zero diagonal)."""
    rng = np.random.default_rng(seed)
    P = rng.uniform([0.0, 0.0], [200.0, 150.0], size=(n, 2))
    D = squareform(pdist(P))
    iu = np.triu_indices(n, 1)
    m = len(iu[0])
    E = rng.normal(0.0, noise, size=m)
    D[iu] += E
    D[(iu[1], iu[0])] += E
    missing = rng.choice(m, size=int(round(frac * m)), replace=False)
    D[iu[0][missing], iu[1][missing]] = np.nan
    D[iu[1][missing], iu[0][missing]] = np.nan
    return P, D


P_19, D_19 = towns_19(6619)       # the region of the notebook: the true positions (km) and the measured distances


def procrustes_error_19(Y, P):
    """The mean distance (km) between the towns of the map Y and their true positions P, once both are centered and Y is
    aligned on P by the best orthogonal transformation (scipy.linalg.orthogonal_procrustes)."""
    Y, P = np.asarray(Y, dtype=float), np.asarray(P, dtype=float)
    Yc, Pc = Y - Y.mean(axis=0), P - P.mean(axis=0)
    R, _ = orthogonal_procrustes(Yc, Pc)
    return float(np.linalg.norm(Yc @ R - Pc, axis=1).mean())

In [ ]:
def map_from_partial_distances_19(D):
    """A map, of shape (n, 2), of the n towns whose measured distances are the matrix D (symmetric, zero diagonal,
    np.nan for a missing pair)."""
    raise NotImplementedError("map_from_partial_distances_19 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M28_19a = answer_M28_19b = ...
with wbm.attempt("M28.19"):          # your function: ⏳ while it is not written
    answer_M28_19a = ...  # a) a number, as computed: procrustes_error_19(your map of D_19, P_19), in km
    answer_M28_19b = ...  # b) a number, as computed: the error (km) of a single classical scaling after filling each missing distance with the mean of the known ones

In [ ]:
check_answer("M28.19b", answer_M28_19b)

HIDDEN_C19 = [(6642, 26), (6644, 30), (6649, 34)]   # three hidden regions of towns_19 (seed, number of towns), renumbered, turned and moved at each run
rng_c19 = np.random.default_rng()       # a new numbering, rotation and translation at each run: no table of values can pass


def map_why_c19(Y, n):
    """'' when Y is a finite map of shape (n, 2), else the reason."""
    arr = as_floats(Y)
    if arr is None or arr.shape != (n, 2):
        return f"attendu une carte de forme ({n}, 2), reçu {None if arr is None else arr.shape}"
    if not np.all(np.isfinite(arr)):
        return "ta carte contient des nombres non finis (nan, inf)"
    return ""


def error_why_c19(error, elapsed):
    """'' when the criterion is met, else the reason."""
    if elapsed >= 10:
        return f"trop lent ({fr(elapsed, 1)} s, il faut moins de 10 s)"
    if error >= 5:
        return (f"{fr(error, 1)} km en moyenne après alignement : un seul double centrage d'une matrice complétée ne suffit "
                "pas ; une carte donne des distances pour toutes les paires")
    if error >= 1:
        return (f"{fr(error, 2)} km en moyenne après alignement, plus de 1 km : répète le cycle davantage, ou pars d'un "
                "meilleur remplissage")
    return ""


with checked("M28.19"):
    results_19 = []
    report_19 = reporter(results_19, stop=False)
    start_19 = time.perf_counter()
    map_c19 = map_from_partial_distances_19(D_19.copy())
    elapsed_19 = time.perf_counter() - start_19
    why_c19 = map_why_c19(map_c19, len(D_19))
    error_c19 = procrustes_error_19(map_c19, P_19) if not why_c19 else float("nan")
    why_c19 = why_c19 or error_why_c19(error_c19, elapsed_19)
    report_19(not why_c19, f"la région du notebook : {fr(error_c19, 3)} km en moyenne après alignement, en "
              f"{fr(elapsed_19, 2)} s", f"la région du notebook : {why_c19}")
    if not filled(answer_M28_19a):
        print("⏳ Ex M28.19a : pas encore fait.")
    elif math.isfinite(error_c19):
        try:
            same_c19 = abs(float(answer_M28_19a) - error_c19) <= 5e-4
        except (TypeError, ValueError):
            same_c19 = False
        report_19(same_c19, "Ex M28.19a : ta réponse est l'erreur de la carte que rend ta fonction",
                  "Ex M28.19a : ta réponse n'est pas l'erreur de la carte que rend ta fonction maintenant, "
                  "procrustes_error_19(map_from_partial_distances_19(D_19), P_19) : exécute à nouveau ta cellule")
    maps_c19 = [as_floats(map_c19)]
    for k_c19, (seed_c19, n_c19) in enumerate(HIDDEN_C19, start=1):
        P_c19, D_c19 = towns_19(seed_c19, n_c19)
        order_c19 = rng_c19.permutation(n_c19)
        angle_c19 = rng_c19.uniform(0, 2 * np.pi)
        turn_c19 = np.array([[np.cos(angle_c19), -np.sin(angle_c19)], [np.sin(angle_c19), np.cos(angle_c19)]])
        P_c19 = P_c19[order_c19] @ turn_c19.T + rng_c19.uniform(-500, 500, size=2)
        D_c19 = D_c19[np.ix_(order_c19, order_c19)]
        start_19 = time.perf_counter()
        got_c19 = map_from_partial_distances_19(D_c19.copy())
        elapsed_19 = time.perf_counter() - start_19
        why_c19 = map_why_c19(got_c19, n_c19)
        if not why_c19 and any(m is not None and m.shape == np.shape(got_c19) and np.allclose(m, got_c19) for m in maps_c19):
            why_c19 = ("ta fonction rend la même carte pour deux régions différentes : elle doit la calculer à partir de la "
                       "matrice reçue")
        error_c19 = procrustes_error_19(got_c19, P_c19) if not why_c19 else float("nan")
        why_c19 = why_c19 or error_why_c19(error_c19, elapsed_19)
        report_19(not why_c19, f"région cachée {k_c19} ({n_c19} villes) : {fr(error_c19, 3)} km en moyenne après "
                  f"alignement, en {fr(elapsed_19, 2)} s", f"région cachée {k_c19} ({n_c19} villes) : {why_c19}")
        maps_c19.append(as_floats(got_c19))
    verdict("M28.19", all(results_19), "critère atteint : moins de 1 km en moyenne après alignement pour les quatre "
            "régions, en moins de 10 secondes chacune.", "critère pas encore atteint : relis les lignes ❌ ci-dessus.")

**Pour réfléchir** : pourquoi un seul positionnement, après un remplissage par la moyenne, se trompe-t-il de plus de 10 km ? Pourquoi l'erreur finale ne descend-elle pas sous l'ordre de grandeur de l'erreur de mesure ? Que se passerait-il si une ville n'avait plus que deux distances connues ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu former une matrice de Gram dans les deux sens, et y lire des distances ?
2. Sais-tu retrouver une carte à partir de distances (double centrage, théorème spectral), à une isométrie près ?
3. Sais-tu faire une montée projetée, et dire ce que change un pas normalisé ?
4. Sais-tu écrire la variation totale comme une forme quadratique, et dire ce qu'elle et un flou font à un motif rapide ?
5. Sais-tu tempérer et tronquer une loi, et dire ce que coûte la troncature ?
6. Sais-tu trouver la prépériode et la période d'une suite sur un ensemble fini ?

**La suite** : CP6, l'examen blanc de la partie VI, puis le chapitre 28 de workbookIA, qui applique ces outils à la visualisation des réseaux, au transfert de style et à la génération de texte.